In [1]:
import sys
!{sys.executable} -m pip install plotly vizro joblib

"""
CIRPA-2026-S1 — Executive BI Dashboard (Vizro)
McKinsey & Company Visual Standard

Design language:
  - White background, minimal gridlines
  - Two-colour palette: McKinsey Blue (#027BFF) + Charcoal (#051C2C)
  - Accent: Light blue (#9FD5FF), Warm alert (#D94F00)
  - Font: Bower / Georgia (serif headings), Helvetica Neue (body)
  - Bold chart titles, direct annotations, no chart chrome
"""

import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px

import vizro.models as vm
import vizro.plotly.express as vpx
from vizro import Vizro
from vizro.managers import data_manager
from vizro.models.types import capture

# =============================================================================
# LOAD DATA
# =============================================================================

DATA_DIR = "/Users/nxmai0309/Documents/My project/Marketing_analytics/outputs"

churn = pd.read_csv(f"{DATA_DIR}/customer_churn_scores.csv")
clv_bin = pd.read_csv(f"{DATA_DIR}/customer_clv_binary_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Bin"})
clv_reg = pd.read_csv(f"{DATA_DIR}/customer_clv_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Reg"})
df_reg_models = pd.read_csv(f"{DATA_DIR}/clv_model_comparison.csv")
clv_model = joblib.load(f"{DATA_DIR}/clv_binary_model.joblib")

df = churn.merge(clv_bin, on="CustomerID").merge(clv_reg, on="CustomerID")

CHURN_BINS = [0, 0.04, 0.25, 0.50, 0.75, 1.01]
CHURN_LABELS = ["<4%", "4–25%", "25–50%", "50–75%", ">75%"]
df["ChurnBand"] = pd.cut(df["Churn_Prob"], bins=CHURN_BINS, labels=CHURN_LABELS, right=False)

TN, FP, FN, TP = 482, 59, 50, 83

data_manager["df_merged"] = df
data_manager["df_reg_models"] = df_reg_models

# =============================================================================
# McKINSEY COLOUR PALETTE
# =============================================================================

MC = {
    "blue":      "#027BFF",   # McKinsey primary blue
    "dark":      "#051C2C",   # Charcoal / near-black
    "light_blue":"#9FD5FF",   # Light accent blue
    "alert":     "#D94F00",   # Warm alert / action red-orange
    "mid_grey":  "#8C8C8C",   # Axis labels, secondary text
    "light_grey":"#D9D9D9",   # Gridlines, borders
    "bg":        "#FFFFFF",   # White background
    "text":      "#051C2C",   # Primary text
    "muted":     "#6B7280",   # Muted annotation text
}

# McKinsey chart layout template
MCK_LAYOUT = dict(
    font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", color=MC["dark"], size=12),
    title=dict(font=dict(family="Georgia, Times New Roman, serif", size=18, color=MC["dark"]), x=0, xanchor="left"),
    plot_bgcolor=MC["bg"],
    paper_bgcolor=MC["bg"],
    xaxis=dict(
        showgrid=False, zeroline=False,
        linecolor=MC["light_grey"], linewidth=1, ticks="outside", ticklen=4,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    yaxis=dict(
        showgrid=True, gridcolor=MC["light_grey"], gridwidth=0.5, griddash="dot",
        zeroline=False, showline=False,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    legend=dict(
        orientation="h", y=-0.15, x=0, xanchor="left",
        font=dict(size=11, color=MC["mid_grey"]),
        bgcolor="rgba(0,0,0,0)", borderwidth=0,
    ),
    margin=dict(t=65, b=55, l=60, r=20),
    hoverlabel=dict(bgcolor=MC["dark"], font_size=11, font_family="Helvetica Neue"),
)

def mck(fig, **overrides):
    """Apply McKinsey layout to any figure."""
    layout = {**MCK_LAYOUT, **overrides}
    fig.update_layout(**layout)
    return fig


# =============================================================================
# CHART FUNCTIONS
# =============================================================================

# ---- PAGE 1: Revenue Concentration ----

@capture("graph")
def graph_1a(data_frame):
    ct = pd.crosstab(data_frame["ChurnBand"], data_frame["CLV_Tier_Bin"])
    for col in ["High", "Not-High"]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[["High", "Not-High"]]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["High"], name="High CLV",
                         marker_color=MC["blue"], text=ct["High"],
                         textposition="inside", textfont=dict(size=12, color="white")))
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["Not-High"], name="Not-High CLV",
                         marker_color=MC["light_blue"], text=ct["Not-High"],
                         textposition="inside", textfont=dict(size=12, color=MC["dark"])))

    total_gt75 = ct.loc[">75%"].sum()
    not_high_gt75 = ct.loc[">75%", "Not-High"]
    fig.add_annotation(x=">75%", y=total_gt75 + 40,
                       text=f"<b>{not_high_gt75} of {total_gt75}</b> are Not-High CLV",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["alert"],
                       font=dict(size=11, color=MC["alert"]), ax=0, ay=-30)

    mck(fig, barmode="stack",
        title=dict(text="Churn risk by CLV tier"),
        xaxis_title="Churn probability band", yaxis_title="Number of customers")
    return fig


# ---- PAGE 2: Segmentation ----

@capture("graph")
def graph_2a(data_frame):
    churn_thresh, clv_thresh = 0.04, 0.5
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=data_frame["Churn_Prob"], y=data_frame["CLV_High_Prob"], mode="markers",
        marker=dict(size=4, opacity=0.4, color=data_frame["CLV_High_Prob"],
                    colorscale=[[0, MC["light_grey"]], [0.5, MC["light_blue"]], [1, MC["blue"]]],
                    showscale=False),
        hovertemplate="Churn: %{x:.2f}<br>CLV prob: %{y:.2f}<extra></extra>", showlegend=False))

    fig.add_hline(y=clv_thresh, line_dash="dash", line_color=MC["mid_grey"], line_width=1)
    fig.add_vline(x=churn_thresh, line_dash="dash", line_color=MC["alert"], line_width=1.5)

    protect = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    nurture = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    monitor = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()
    depri = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()

    for lx, ly, txt, col in [
        (0.02, 0.95, f"PROTECT  {protect}", MC["blue"]),
        (0.55, 0.95, f"NURTURE  {nurture}", MC["alert"]),
        (0.02, 0.05, f"MONITOR  {monitor}", MC["mid_grey"]),
        (0.55, 0.05, f"DEPRIORITISE  {depri}", MC["muted"]),
    ]:
        fig.add_annotation(x=lx, y=ly, text=f"<b>{txt}</b>", showarrow=False,
                           font=dict(size=12, color=col, family="Helvetica Neue"), xref="x", yref="y")

    mck(fig, title=dict(text="Retention priority matrix — every customer positioned"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02], showgrid=False),
        yaxis=dict(title="CLV high probability", range=[-0.02, 1.02], showgrid=False))
    return fig


@capture("graph")
def graph_2b(data_frame):
    bin_counts = data_frame["CLV_Tier_Bin"].value_counts()
    reg_counts = data_frame["CLV_Tier_Reg"].value_counts()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=["High", "Not-High"],
                         y=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         name="Binary model", marker_color=MC["blue"],
                         text=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))
    fig.add_trace(go.Bar(x=["High", "Mid", "Low"],
                         y=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         name="Regression model", marker_color=MC["light_blue"],
                         text=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))

    gap = abs(bin_counts.get("High", 0) - reg_counts.get("High", 0))
    fig.add_annotation(x=0.98, y=0.95, xref="paper", yref="paper",
                       text=f"<b>{gap}</b> customers in<br>disagreement zone",
                       showarrow=False, font=dict(size=11, color=MC["alert"]),
                       align="right")

    mck(fig, barmode="group", title=dict(text="CLV tier distribution — binary vs regression"),
        xaxis_title="CLV tier", yaxis_title="Number of customers")
    return fig


# ---- PAGE 3: Churn Prediction ----

@capture("graph")
def graph_3a(data_frame):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=data_frame["Churn_Prob"], nbinsx=60,
                               marker_color=MC["blue"], opacity=0.85, name="Customers"))

    n_opt = (data_frame["Churn_Prob"] >= 0.04).sum()
    n_def = (data_frame["Churn_Prob"] >= 0.50).sum()
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>Optimal 0.04</b><br>{n_opt:,} flagged",
                  annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"Default 0.50<br>{n_def:,} flagged",
                  annotation_position="top left",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, title=dict(text="Why the threshold is 0.04, not 0.50"),
        xaxis_title="Churn probability", yaxis_title="Number of customers", showlegend=False)
    return fig


@capture("graph")
def graph_3b(data_frame):
    sorted_probs = np.sort(data_frame["Churn_Prob"].values)
    cdf = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sorted_probs, y=cdf, mode="lines",
                             line=dict(color=MC["blue"], width=2.5), name="CDF",
                             fill="tozeroy", fillcolor="rgba(2,123,255,0.06)"))

    pct_004 = (data_frame["Churn_Prob"] < 0.04).mean() * 100
    pct_050 = (data_frame["Churn_Prob"] < 0.50).mean() * 100
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>0.04</b> → {pct_004:.1f}% below",
                  annotation_position="top right", annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"0.50 → {pct_050:.1f}% below",
                  annotation_position="top left", annotation_font=dict(color=MC["mid_grey"], size=10))
    fig.add_hline(y=pct_004, line_dash="dot", line_color=MC["alert"], line_width=1, opacity=0.4)

    mck(fig, title=dict(text="Churn probability CDF — almost everyone is at risk"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02]),
        yaxis=dict(title="% of customers (cumulative)", range=[0, 105]),
        showlegend=False)
    return fig


# ---- PAGE 4: Why Customers Churn ----

@capture("graph")
def graph_4a(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["Churn_Prob"]
        fig.add_trace(go.Box(y=subset, name=f"{tier} CLV", marker_color=color,
                             line=dict(color=MC["dark"], width=1.5), boxmean=False,
                             fillcolor=color))

    medians = {t: data_frame[data_frame["CLV_Tier_Reg"] == t]["Churn_Prob"].median() for t in tier_order}
    for i, (tier, med) in enumerate(medians.items()):
        fig.add_annotation(x=i, y=med - 0.06, text=f"<b>{med:.3f}</b>", showarrow=False,
                           font=dict(size=11, color=MC["dark"]))

    mck(fig, title=dict(text="Churn risk by CLV tier — value predicts retention"),
        yaxis=dict(title="Churn probability", range=[-0.05, 1.05]),
        xaxis_title="CLV tier (regression model)", showlegend=False)
    return fig


# ---- PAGE 5: CLV Intelligence ----

@capture("graph")
def graph_5a_reg(data_frame):
    models = data_frame["Model"].tolist()
    cv_r2 = data_frame["CV R² Mean"].tolist()
    holdout_r2 = data_frame["Holdout R²"].tolist()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=models, y=cv_r2, name="CV R² (5-fold mean)", marker_color=MC["dark"],
                         text=[f"{v:.3f}" for v in cv_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_trace(go.Bar(x=models, y=holdout_r2, name="Holdout R²", marker_color=MC["blue"],
                         text=[f"{v:.3f}" for v in holdout_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_hline(y=0.50, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text="<b>QG3 threshold (R² ≥ 0.50)</b>", annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10, family="Helvetica Neue"))

    train_r2 = data_frame.iloc[0]["Train R²"]
    overfit_gap = data_frame.iloc[0]["Overfit Gap"]
    fig.add_annotation(x="Ridge (baseline)", y=holdout_r2[0] + 0.04,
                       text=f"Train R² = {train_r2:.3f}<br>Gap: {overfit_gap:+.3f}",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["mid_grey"],
                       font=dict(size=10, color=MC["muted"]), ax=75, ay=-40)

    mck(fig, barmode="group", title=dict(text="Why regression failed — no model met R² ≥ 0.50"),
        yaxis=dict(title="R² score", range=[-0.05, 0.65], zeroline=True, zerolinecolor=MC["light_grey"]),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5b_reg(data_frame):
    models = data_frame["Model"].tolist()
    top5_mae = data_frame["Top-5% MAE"].tolist()
    min_idx = int(np.argmin(top5_mae))
    colors = [MC["light_blue"]] * len(models)
    colors[min_idx] = MC["blue"]

    fig = go.Figure(go.Bar(x=models, y=top5_mae, marker_color=colors,
                           text=[f"£{v:,.0f}" for v in top5_mae], textposition="outside",
                           textfont=dict(size=12, color=MC["dark"]), width=0.5))
    fig.add_annotation(x=models[min_idx], y=top5_mae[min_idx] - 800,
                       text="<b>✓ Lowest error</b>", showarrow=False,
                       font=dict(size=10, color=MC["blue"]))

    mck(fig, title=dict(text="Prediction error on top-5% customers (MAE)"),
        yaxis=dict(title="Mean absolute error (£)", tickformat=",.0f", tickprefix="£"),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5c_reg(data_frame):
    total = TN + FP + FN + TP
    accuracy = (TP + TN) / total
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * precision * recall / (precision + recall)

    metrics = ["Accuracy", "Recall", "Precision", "F1 Score"]
    values = [accuracy, recall, precision, f1]
    colors = [MC["blue"], MC["blue"], MC["light_blue"], MC["light_blue"]]

    fig = go.Figure(go.Bar(y=metrics, x=values, orientation="h", marker_color=colors,
                           text=[f"<b>{v:.1%}</b>" for v in values], textposition="outside",
                           textfont=dict(size=13, color=MC["dark"])))
    mck(fig, title=dict(text="Binary CLV classifier — Random Forest performance"),
        xaxis=dict(title="Score", range=[0, 1.05], tickformat=".0%"),
        margin=dict(t=65, b=55, l=95, r=20))
    return fig


@capture("graph")
def graph_5d_reg(data_frame):
    feature_names = list(clv_model.feature_names_in_)
    importances = list(clv_model.feature_importances_)
    df_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=True)
    top3 = df_imp.nlargest(3, "Importance")["Feature"].tolist()
    colors = [MC["blue"] if f in top3 else MC["light_blue"] for f in df_imp["Feature"]]

    fig = go.Figure(go.Bar(y=df_imp["Feature"], x=df_imp["Importance"], orientation="h",
                           marker_color=colors, text=[f"{v:.1%}" for v in df_imp["Importance"]],
                           textposition="outside", textfont=dict(size=11, color=MC["dark"])))
    mck(fig, title=dict(text="CLV feature importance — Random Forest classifier"),
        xaxis=dict(title="Gini importance", tickformat=".0%"),
        margin=dict(t=65, b=55, l=170, r=20))
    return fig


@capture("graph")
def graph_5a(data_frame):
    ct = pd.crosstab(data_frame["CLV_Tier_Bin"], data_frame["CLV_Tier_Reg"])
    ct = ct.reindex(index=["High", "Not-High"], columns=["High", "Mid", "Low"], fill_value=0)

    fig = go.Figure(go.Heatmap(
        z=ct.values, x=["High", "Mid", "Low"], y=[f"Binary: {r}" for r in ["High", "Not-High"]],
        colorscale=[[0, MC["bg"]], [0.3, MC["light_blue"]], [1.0, MC["blue"]]],
        text=[[f"<b>{v}</b>" for v in row] for row in ct.values], texttemplate="%{text}",
        textfont=dict(size=18, color=MC["dark"]),
        hovertemplate="Binary: %{y}<br>Regression: %{x}<br>Count: %{z}<extra></extra>",
        showscale=False))

    mck(fig, title=dict(text="CLV model agreement — where do the models agree?"),
        xaxis_title="Regression CLV tier", yaxis_title="",
        margin=dict(t=65, b=55, l=120, r=20))
    return fig


@capture("graph")
def graph_5b(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["CLV_High_Prob"]
        fig.add_trace(go.Histogram(x=subset, nbinsx=50, histnorm="probability density",
                                   name=f"{tier} (n={len(subset)})", marker_color=color, opacity=0.55))
    fig.add_vline(x=0.5, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text="Binary threshold", annotation_position="top right",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, barmode="overlay", title=dict(text="Binary CLV probability by regression tier"),
        xaxis=dict(title="CLV high probability", range=[-0.02, 1.05]),
        yaxis_title="Density")
    return fig


# ---- PAGE 6: ROI ----

@capture("graph")
def graph_6a(data_frame):
    ct = data_frame.groupby(["ChurnBand", "CLV_Tier_Bin"]).agg(
        Count=("CustomerID", "count"), MeanCLVProb=("CLV_High_Prob", "mean"),
        MeanChurnProb=("Churn_Prob", "mean")).reset_index()
    ct["RiskIndex"] = ct["Count"] * ct["MeanCLVProb"] * ct["MeanChurnProb"]

    fig = go.Figure()
    for tier, color in [("High", MC["blue"]), ("Not-High", MC["light_blue"])]:
        subset = ct[ct["CLV_Tier_Bin"] == tier]
        fig.add_trace(go.Bar(x=subset["ChurnBand"].astype(str), y=subset["RiskIndex"],
                             name=f"{tier} CLV", marker_color=color,
                             text=[f"{v:.0f}" for v in subset["RiskIndex"]],
                             textposition="outside", textfont=dict(size=11, color=MC["dark"])))

    key_band = ct[(ct["ChurnBand"] == "4–25%") & (ct["CLV_Tier_Bin"] == "High")]
    if len(key_band) > 0:
        fig.add_annotation(x="4–25%", y=key_band.iloc[0]["RiskIndex"] + 3,
                           text=f"<b>Core opportunity:</b><br>{int(key_band.iloc[0]['Count'])} High-CLV customers",
                           showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["blue"],
                           font=dict(size=10, color=MC["blue"]), ax=65, ay=-30)

    mck(fig, barmode="group", title=dict(text="Where is the money most at risk?"),
        xaxis_title="Churn probability band", yaxis_title="Revenue-at-risk index")
    return fig


# =============================================================================
# PAGES — Business Insights Integrated
# =============================================================================

page_1 = vm.Page(
    title="Revenue Concentration",
    components=[
        vm.Container(
            title="Insight 1.1 — The Business Is Dangerously Concentrated",
            components=[
                vm.Graph(figure=graph_1a(data_frame="df_merged")),
                vm.Card(text="""
The top 1% of customers (~43 accounts) generate **~31.8% of total revenue**. The top 5% generate ~50.4%. \
The Gini coefficient of **0.716** confirms this is not a healthy long-tail — it is a business where a \
handful of accounts constitute an existential dependency.

**345 of 347** customers with >75% churn probability are Not-High CLV. Revenue concentration becomes \
risk concentration — you are not losing your valuable customers to churn, you are losing your low-value ones.

**Decision enabled:** The CFO must treat the top 43 accounts as a separate risk category with dedicated \
retention resource. Losing three in a single quarter is a material revenue event.
"""),
            ],
        ),
        vm.Container(
            title="Insight 1.2 — The Revenue-at-Risk Exposure Is Quantified",
            components=[
                vm.Card(text="""
The **Protect quadrant** contains 140 customers (4.2% of the base) holding **£546,537** in predicted \
lifetime value — 33.6% of total predicted CLV. At mean churn probability of 11.6%, expected unmitigated \
loss is **~£63,537**.

The break-even retention rate at £10/customer is **0.3%**. At £25 it is 0.6%. At £50 it is 1.3%. The \
programme covers its own cost if it retains fewer than **2 customers in every 150 targeted**.

Losing the top 10 customers costs **£1,537,659** (17.3% of revenue). Losing the top 50 costs £2,959,342 (33.3%).

**Decision enabled:** A programme costing £50,000 annually that prevents the loss of two top-10 accounts \
pays for itself **14× over**. The cost of the programme is not the risk. The cost of inaction is.
"""),
            ],
        ),
    ],
)

page_2 = vm.Page(
    title="Segmentation",
    components=[
        vm.Container(
            title="Insight 2.1 — Four Segments Require Four Different Strategies",
            components=[
                vm.Graph(figure=graph_2a(data_frame="df_merged")),
                vm.Card(text="""
Every customer positioned by churn risk (x-axis) and CLV probability (y-axis). Four quadrants at \
churn threshold 0.04 (profit-optimised) and CLV probability 0.50:

**Protect** (104) — High CLV, low churn. £546K predicted CLV. Dedicated account ownership. \
**Nurture** (657) — High CLV, at risk. Primary retention target at £582K. \
**Monitor** (1) — Low CLV, safe. Digital-only. \
**Deprioritise** (2,608) — Low CLV, high churn. Zero high-touch spend justified.

**Decision enabled:** Budget allocation is proportional to CLV at stake per quadrant, not customer count.
"""),
            ],
        ),
        vm.Container(
            title="Insight 2.2 — Model Disagreement Defines the Monitoring Zone",
            components=[
                vm.Graph(figure=graph_2b(data_frame="df_merged")),
                vm.Card(text="""
Binary model: **761 High**, 2,609 Not-High. Regression model: **603 High**, 570 Mid, 2,197 Low. \
Model disagreement on **158 customers** — the honest acknowledgment of uncertainty.

**513 customers** classified High by both models are confident Protect/Nurture candidates. \
The 158 in disagreement enter a watch-list workflow with lighter-touch intervention.
"""),
            ],
        ),
    ],
)

page_3 = vm.Page(
    title="Churn Prediction",
    components=[
        vm.Container(
            title="Insight 3.3 — The Optimal Threshold and Its Business Value",
            components=[
                vm.Graph(figure=graph_3a(data_frame="df_merged")),
                vm.Card(text="""
The default 50% threshold is the **wrong operational threshold**. At the profit-maximising threshold \
of **4%**: 3,265 flagged. Expected revenue recovered **£187,877**. Net value **£168,257**. \
ROI approximately **8.6:1** at midpoint assumptions.

**Decision enabled:** Set the CRM churn alert trigger at **0.04 probability**. The difference between \
4% and 50% is ~£168,000 in expected net value per cycle.
"""),
            ],
        ),
        vm.Container(
            title="Insight 3.1 — Almost Everyone Is At Risk (With an Honest Caveat)",
            components=[
                vm.Graph(figure=graph_3b(data_frame="df_merged")),
                vm.Card(text="""
Only **3.1%** of customers have churn probability below 0.04 — 96.9% are technically at risk. \
In non-contractual retail, this is behaviourally accurate.

**The honest caveat:** AUC-ROC of **0.755** — below the QG3 target of 0.80. The model is a ranked \
prioritisation tool, not a certainty engine. Every decision should be treated as a strong signal \
requiring human judgment.
"""),
            ],
        ),
    ],
)

page_4 = vm.Page(
    title="Why Customers Churn",
    components=[
        vm.Container(
            title="Insight 4.1 — CLV Tier Is the Strongest Natural Separator of Churn Risk",
            components=[
                vm.Graph(figure=graph_4a(data_frame="df_merged")),
                vm.Card(text="""
High CLV median churn: **0.196**. Mid: **0.403**. Low: **0.608**. The boxes barely overlap.

Global SHAP: **Monetary** (32.7%), **Frequency** (25.0%), **AverageRevenuePerOrder** (18.5%), \
**UniqueItemsPerOrder** (11.1%), **Recency** (9.6%) — 96.8% of total signal.

**Decision enabled:** The CRM health metric is a composite led by Monetary trajectory and Frequency \
decline, not a single recency flag.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.2 — Churn Drivers Differ Materially Across Segments",
            components=[
                vm.Card(text="""
**Platinum churn** — Relationship management problem. B2B wholesale accounts with InterPurchaseInterval \
of 46 days. Signal: cycle deviation + Frequency decline. **Owner: Sales function.**

**Silver churn** — Engagement deepening problem. Recency 87 days, near-zero volatility. Trigger: \
customer crossing **60 days** post-last-purchase. **Owner: CRM & Loyalty.**

**Bronze churn** — Conversion failure. Recency 159 days, one-time buyers. Intervention window closed. \
**Owner: Low-cost digital or deprioritise entirely.**

**Decision enabled:** Route alerts by segment. Platinum/Gold → account management. Silver → CRM. \
Bronze → automated minimum-cost or no intervention.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.4 — There Is a Specific Deadline for Intervention",
            components=[
                vm.Card(text="""
**Intervention deadlines:** Platinum CRM trigger at **75 days** (10–15 day window before 90-day threshold). \
Silver trigger at **60 days**. Beyond **120 days** any segment: reclassify as win-back, lower-cost workflow.

A Platinum customer at 85 days is in the retention window. At 95 days, statistically in re-activation — \
higher cost, lower success probability.
"""),
            ],
        ),
    ],
)

page_5 = vm.Page(
    title="CLV Intelligence",
    components=[
        vm.Container(
            title="Insight 5.1 — Regression Failed: Binary Classification Is the Fallback",
            components=[
                vm.Graph(figure=graph_5a_reg(data_frame="df_reg_models")),
                vm.Card(text="""
No model met **R² ≥ 0.50** (QG3). Ridge Train R² = 0.006. XGBoost degraded to 0.133 on holdout. \
Binary classification adopted — a **data-justified decision**.

**Decision enabled:** Remove historical Monetary as primary ranking metric. Use predicted CLV from \
the binary model for all budget allocation.
"""),
            ],
        ),
        vm.Container(
            title="Where Regression Still Matters — Top-5% Accuracy",
            components=[
                vm.Graph(figure=graph_5b_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Random Forest: **£13,600 MAE** on top-5%. Overall R² is the wrong metric when your business case \
concentrates on 5% of customers. Use the binary classifier for tier assignment; use the RF regressor \
for **within-tier ranking** of Protect accounts.
"""),
            ],
        ),
        vm.Container(
            title="Binary CLV Classifier — Production Performance",
            components=[
                vm.Graph(figure=graph_5c_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Accuracy **83.8%**. Recall **62.4%**. Precision **58.5%**. CLV threshold: training set median **£1,810**.

Per-customer spend ceilings: Protect **£780**, Nurture loyalty investment justified, Monitor **£10–£25**, \
Deprioritise **no spend above £10**.
"""),
            ],
        ),
        vm.Container(
            title="CLV Feature Importance",
            components=[
                vm.Graph(figure=graph_5d_reg(data_frame="df_reg_models")),
                vm.Card(text="""
**Monetary (31.1%)** dominates. Same 5 features drive both churn and CLV models — confirming \
feature engineering was sound. Any Protect/Nurture customer with return rate >25% needs CLV adjusted: \
CLV × (1 − return rate).
"""),
            ],
        ),
        vm.Container(
            title="Insight 5.2 — CLV Distribution Across the Priority Matrix",
            components=[
                vm.Graph(figure=graph_5a(data_frame="df_merged")),
                vm.Card(text="""
**513** classified High by both models. Protect:Deprioritise CLV ratio is **25:1**.

| Quadrant | Customers | Mean CLV | Total CLV | % Forward Rev |
|---|---|---|---|---|
| **Protect** | 140 | £3,904 | £546,537 | 33.6% |
| **Nurture** | 57 | £10,220 | £582,517 | 35.8% |
| **Deprioritise** | 3,168 | £156 | £494,953 | 30.4% |
"""),
            ],
        ),
        vm.Container(
            title="Model Consistency Check",
            components=[
                vm.Graph(figure=graph_5b(data_frame="df_merged")),
                vm.Card(text="""
High-regression-tier customers cluster at high CLV_High_Prob. Clean separation validates the \
binary fallback. Overlap in **0.3–0.7** quantifies genuine uncertainty — the 158 disagreement customers.
"""),
            ],
        ),
    ],
)

page_6 = vm.Page(
    title="ROI",
    components=[
        vm.Container(
            title="Insight 6.1 — Where Is the Money Most at Risk?",
            components=[
                vm.Graph(figure=graph_6a(data_frame="df_merged")),
                vm.Card(text="""
**336 High-CLV customers** in the 4–25% churn band = core retention opportunity.

ROI range: **7.8:1** (conservative) to **117.1:1** (optimistic). Midpoint **31.2:1**. \
Every scenario is ROI-positive. Budget: 140 × £25 = **£3,500**. Expected preserved: **£109,307**.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.2 — The Cost of Doing Nothing Is £63,573",
            components=[
                vm.Card(text="""
Expected unmitigated loss: **£63,573** (Protect CLV £546,537 × 11.6% churn probability). \
Midpoint programme costs **£3,500**. Net expected value: **£105,807**.

Total all-tier investment: **£4,975**. Combined CLV at stake: **£1,129,054**. \
Success metric: if Protect retention exceeds **1.3%**, programme has paid for itself at highest cost.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.3 — Stop Spending on the Deprioritise Quadrant",
            components=[
                vm.Card(text="""
Deprioritise: 3,168 customers, mean CLV £156, 53.9% churn. At £25/customer and 10% retention: \
**loss of £9.38 per customer**.

Current undifferentiated spend on Deprioritise: **£79,200**. Reallocate to 140 Protect customers → \
per-customer spend rises to **£591**, funding dedicated account management. Revenue preserved unchanged \
at £109,307. **This is a reallocation instruction, not a budget increase.**
"""),
            ],
        ),
    ],
)


# =============================================================================
# DASHBOARD ASSEMBLY
# =============================================================================

dashboard = vm.Dashboard(
    title="CIRPA — Customer Intelligence & Retention Analytics",
    pages=[page_1, page_2, page_3, page_4, page_5, page_6],
    theme="vizro_light",
    navigation=vm.Navigation(
        nav_selector=vm.NavBar(
            items=[
                vm.NavLink(icon="bar_chart", label="Revenue", pages=["Revenue Concentration"]),
                vm.NavLink(icon="groups", label="Segments", pages=["Segmentation"]),
                vm.NavLink(icon="warning", label="Churn", pages=["Churn Prediction"]),
                vm.NavLink(icon="psychology", label="Drivers", pages=["Why Customers Churn"]),
                vm.NavLink(icon="payments", label="CLV", pages=["CLV Intelligence"]),
                vm.NavLink(icon="trending_up", label="ROI", pages=["ROI"]),
            ]
        )
    ),
)

Vizro().build(dashboard).run(port=8050, jupyter_mode="external")

"""
CIRPA-2026-S1 — Executive BI Dashboard (Vizro)
McKinsey & Company Visual Standard

Design language:
  - White background, minimal gridlines
  - Two-colour palette: McKinsey Blue (#027BFF) + Charcoal (#051C2C)
  - Accent: Light blue (#9FD5FF), Warm alert (#D94F00)
  - Font: Bower / Georgia (serif headings), Helvetica Neue (body)
  - Bold chart titles, direct annotations, no chart chrome
"""

import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px

import vizro.models as vm
import vizro.plotly.express as vpx
from vizro import Vizro
from vizro.managers import data_manager
from vizro.models.types import capture

# =============================================================================
# LOAD DATA
# =============================================================================


DATA_DIR = "/Users/nxmai0309/Documents/My project/Marketing_analytics/outputs"

churn = pd.read_csv(f"{DATA_DIR}/customer_churn_scores.csv")
clv_bin = pd.read_csv(f"{DATA_DIR}/customer_clv_binary_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Bin"})
clv_reg = pd.read_csv(f"{DATA_DIR}/customer_clv_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Reg"})
df_reg_models = pd.read_csv(f"{DATA_DIR}/clv_model_comparison.csv")
clv_model = joblib.load(f"{DATA_DIR}/clv_binary_model.joblib")

df = churn.merge(clv_bin, on="CustomerID").merge(clv_reg, on="CustomerID")

CHURN_BINS = [0, 0.04, 0.25, 0.50, 0.75, 1.01]
CHURN_LABELS = ["<4%", "4–25%", "25–50%", "50–75%", ">75%"]
df["ChurnBand"] = pd.cut(df["Churn_Prob"], bins=CHURN_BINS, labels=CHURN_LABELS, right=False)

TN, FP, FN, TP = 482, 59, 50, 83

data_manager["df_merged"] = df
data_manager["df_reg_models"] = df_reg_models

# =============================================================================
# McKINSEY COLOUR PALETTE
# =============================================================================

MC = {
    "blue":      "#027BFF",   # McKinsey primary blue
    "dark":      "#051C2C",   # Charcoal / near-black
    "light_blue":"#9FD5FF",   # Light accent blue
    "alert":     "#D94F00",   # Warm alert / action red-orange
    "mid_grey":  "#8C8C8C",   # Axis labels, secondary text
    "light_grey":"#D9D9D9",   # Gridlines, borders
    "bg":        "#FFFFFF",   # White background
    "text":      "#051C2C",   # Primary text
    "muted":     "#6B7280",   # Muted annotation text
}

# McKinsey chart layout template
MCK_LAYOUT = dict(
    font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", color=MC["dark"], size=12),
    title=dict(font=dict(family="Georgia, Times New Roman, serif", size=18, color=MC["dark"]), x=0, xanchor="left"),
    plot_bgcolor=MC["bg"],
    paper_bgcolor=MC["bg"],
    xaxis=dict(
        showgrid=False, zeroline=False,
        linecolor=MC["light_grey"], linewidth=1, ticks="outside", ticklen=4,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    yaxis=dict(
        showgrid=True, gridcolor=MC["light_grey"], gridwidth=0.5, griddash="dot",
        zeroline=False, showline=False,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    legend=dict(
        orientation="h", y=-0.15, x=0, xanchor="left",
        font=dict(size=11, color=MC["mid_grey"]),
        bgcolor="rgba(0,0,0,0)", borderwidth=0,
    ),
    margin=dict(t=65, b=55, l=60, r=20),
    height=520,
    hoverlabel=dict(bgcolor=MC["dark"], font_size=11, font_family="Helvetica Neue"),
)

def mck(fig, **overrides):
    """Apply McKinsey layout to any figure."""
    layout = {**MCK_LAYOUT, **overrides}
    fig.update_layout(**layout)
    return fig


# =============================================================================
# CHART FUNCTIONS
# =============================================================================

# ---- PAGE 1: Revenue Concentration ----

@capture("graph")
def graph_1a(data_frame):
    ct = pd.crosstab(data_frame["ChurnBand"], data_frame["CLV_Tier_Bin"])
    for col in ["High", "Not-High"]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[["High", "Not-High"]]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["High"], name="High CLV",
                         marker_color=MC["blue"], text=ct["High"],
                         textposition="inside", textfont=dict(size=12, color="white")))
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["Not-High"], name="Not-High CLV",
                         marker_color=MC["light_blue"], text=ct["Not-High"],
                         textposition="inside", textfont=dict(size=12, color=MC["dark"])))

    total_gt75 = ct.loc[">75%"].sum()
    not_high_gt75 = ct.loc[">75%", "Not-High"]
    fig.add_annotation(x=">75%", y=total_gt75 + 40,
                       text=f"<b>{not_high_gt75} of {total_gt75}</b> are Not-High CLV",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["alert"],
                       font=dict(size=11, color=MC["alert"]), ax=0, ay=-30)

    mck(fig, barmode="stack",
        title=dict(text="Churn risk by CLV tier"),
        xaxis_title="Churn probability band", yaxis_title="Number of customers")
    return fig


# ---- PAGE 2: Segmentation ----

@capture("graph")
def graph_2a(data_frame):
    churn_thresh, clv_thresh = 0.04, 0.5
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=data_frame["Churn_Prob"], y=data_frame["CLV_High_Prob"], mode="markers",
        marker=dict(size=4, opacity=0.4, color=data_frame["CLV_High_Prob"],
                    colorscale=[[0, MC["light_grey"]], [0.5, MC["light_blue"]], [1, MC["blue"]]],
                    showscale=False),
        hovertemplate="Churn: %{x:.2f}<br>CLV prob: %{y:.2f}<extra></extra>", showlegend=False))

    fig.add_hline(y=clv_thresh, line_dash="dash", line_color=MC["mid_grey"], line_width=1)
    fig.add_vline(x=churn_thresh, line_dash="dash", line_color=MC["alert"], line_width=1.5)

    protect = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    nurture = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    monitor = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()
    depri = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()

    for lx, ly, txt, col in [
        (0.02, 0.95, f"PROTECT  {protect}", MC["blue"]),
        (0.55, 0.95, f"NURTURE  {nurture}", MC["alert"]),
        (0.02, 0.05, f"MONITOR  {monitor}", MC["mid_grey"]),
        (0.55, 0.05, f"DEPRIORITISE  {depri}", MC["muted"]),
    ]:
        fig.add_annotation(x=lx, y=ly, text=f"<b>{txt}</b>", showarrow=False,
                           font=dict(size=12, color=col, family="Helvetica Neue"), xref="x", yref="y")

    mck(fig, title=dict(text="Retention priority matrix — every customer positioned"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02], showgrid=False),
        yaxis=dict(title="CLV high probability", range=[-0.02, 1.02], showgrid=False))
    return fig


@capture("graph")
def graph_2b(data_frame):
    bin_counts = data_frame["CLV_Tier_Bin"].value_counts()
    reg_counts = data_frame["CLV_Tier_Reg"].value_counts()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=["High", "Not-High"],
                         y=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         name="Binary model", marker_color=MC["blue"],
                         text=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))
    fig.add_trace(go.Bar(x=["High", "Mid", "Low"],
                         y=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         name="Regression model", marker_color=MC["light_blue"],
                         text=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))

    gap = abs(bin_counts.get("High", 0) - reg_counts.get("High", 0))
    fig.add_annotation(x=0.98, y=0.95, xref="paper", yref="paper",
                       text=f"<b>{gap}</b> customers in<br>disagreement zone",
                       showarrow=False, font=dict(size=11, color=MC["alert"]),
                       align="right")

    mck(fig, barmode="group", title=dict(text="CLV tier distribution — binary vs regression"),
        xaxis_title="CLV tier", yaxis_title="Number of customers")
    return fig


# ---- PAGE 3: Churn Prediction ----

@capture("graph")
def graph_3a(data_frame):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=data_frame["Churn_Prob"], nbinsx=60,
                               marker_color=MC["blue"], opacity=0.85, name="Customers"))

    n_opt = (data_frame["Churn_Prob"] >= 0.04).sum()
    n_def = (data_frame["Churn_Prob"] >= 0.50).sum()
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>Optimal 0.04</b><br>{n_opt:,} flagged",
                  annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"Default 0.50<br>{n_def:,} flagged",
                  annotation_position="top left",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, title=dict(text="Why the threshold is 0.04, not 0.50"),
        xaxis_title="Churn probability", yaxis_title="Number of customers", showlegend=False)
    return fig


@capture("graph")
def graph_3b(data_frame):
    sorted_probs = np.sort(data_frame["Churn_Prob"].values)
    cdf = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sorted_probs, y=cdf, mode="lines",
                             line=dict(color=MC["blue"], width=2.5), name="CDF",
                             fill="tozeroy", fillcolor="rgba(2,123,255,0.06)"))

    pct_004 = (data_frame["Churn_Prob"] < 0.04).mean() * 100
    pct_050 = (data_frame["Churn_Prob"] < 0.50).mean() * 100
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>0.04</b> → {pct_004:.1f}% below",
                  annotation_position="top right", annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"0.50 → {pct_050:.1f}% below",
                  annotation_position="top left", annotation_font=dict(color=MC["mid_grey"], size=10))
    fig.add_hline(y=pct_004, line_dash="dot", line_color=MC["alert"], line_width=1, opacity=0.4)

    mck(fig, title=dict(text="Churn probability CDF — almost everyone is at risk"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02]),
        yaxis=dict(title="% of customers (cumulative)", range=[0, 105]),
        showlegend=False)
    return fig


# ---- PAGE 4: Why Customers Churn ----

@capture("graph")
def graph_4a(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["Churn_Prob"]
        fig.add_trace(go.Box(y=subset, name=f"{tier} CLV", marker_color=color,
                             line=dict(color=MC["dark"], width=1.5), boxmean=False,
                             fillcolor=color))

    medians = {t: data_frame[data_frame["CLV_Tier_Reg"] == t]["Churn_Prob"].median() for t in tier_order}
    for i, (tier, med) in enumerate(medians.items()):
        fig.add_annotation(x=i, y=med - 0.06, text=f"<b>{med:.3f}</b>", showarrow=False,
                           font=dict(size=11, color=MC["dark"]))

    mck(fig, title=dict(text="Churn risk by CLV tier — value predicts retention"),
        yaxis=dict(title="Churn probability", range=[-0.05, 1.05]),
        xaxis_title="CLV tier (regression model)", showlegend=False)
    return fig


# ---- PAGE 5: CLV Intelligence ----

@capture("graph")
def graph_5a_reg(data_frame):
    models = data_frame["Model"].tolist()
    cv_r2 = data_frame["CV R² Mean"].tolist()
    holdout_r2 = data_frame["Holdout R²"].tolist()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=models, y=cv_r2, name="CV R² (5-fold mean)", marker_color=MC["dark"],
                         text=[f"{v:.3f}" for v in cv_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_trace(go.Bar(x=models, y=holdout_r2, name="Holdout R²", marker_color=MC["blue"],
                         text=[f"{v:.3f}" for v in holdout_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_hline(y=0.50, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text="<b>QG3 threshold (R² ≥ 0.50)</b>", annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10, family="Helvetica Neue"))

    train_r2 = data_frame.iloc[0]["Train R²"]
    overfit_gap = data_frame.iloc[0]["Overfit Gap"]
    fig.add_annotation(x="Ridge (baseline)", y=holdout_r2[0] + 0.04,
                       text=f"Train R² = {train_r2:.3f}<br>Gap: {overfit_gap:+.3f}",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["mid_grey"],
                       font=dict(size=10, color=MC["muted"]), ax=75, ay=-40)

    mck(fig, barmode="group", title=dict(text="Why regression failed — no model met R² ≥ 0.50"),
        yaxis=dict(title="R² score", range=[-0.05, 0.65], zeroline=True, zerolinecolor=MC["light_grey"]),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5b_reg(data_frame):
    models = data_frame["Model"].tolist()
    top5_mae = data_frame["Top-5% MAE"].tolist()
    min_idx = int(np.argmin(top5_mae))
    colors = [MC["light_blue"]] * len(models)
    colors[min_idx] = MC["blue"]

    fig = go.Figure(go.Bar(x=models, y=top5_mae, marker_color=colors,
                           text=[f"£{v:,.0f}" for v in top5_mae], textposition="outside",
                           textfont=dict(size=12, color=MC["dark"]), width=0.5))
    fig.add_annotation(x=models[min_idx], y=top5_mae[min_idx] - 800,
                       text="<b>✓ Lowest error</b>", showarrow=False,
                       font=dict(size=10, color=MC["blue"]))

    mck(fig, title=dict(text="Prediction error on top-5% customers (MAE)"),
        yaxis=dict(title="Mean absolute error (£)", tickformat=",.0f", tickprefix="£"),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5c_reg(data_frame):
    total = TN + FP + FN + TP
    accuracy = (TP + TN) / total
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * precision * recall / (precision + recall)

    metrics = ["Accuracy", "Recall", "Precision", "F1 Score"]
    values = [accuracy, recall, precision, f1]
    colors = [MC["blue"], MC["blue"], MC["light_blue"], MC["light_blue"]]

    fig = go.Figure(go.Bar(y=metrics, x=values, orientation="h", marker_color=colors,
                           text=[f"<b>{v:.1%}</b>" for v in values], textposition="outside",
                           textfont=dict(size=13, color=MC["dark"])))
    mck(fig, title=dict(text="Binary CLV classifier — Random Forest performance"),
        xaxis=dict(title="Score", range=[0, 1.05], tickformat=".0%"),
        margin=dict(t=65, b=55, l=95, r=20))
    return fig


@capture("graph")
def graph_5d_reg(data_frame):
    feature_names = list(clv_model.feature_names_in_)
    importances = list(clv_model.feature_importances_)
    df_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=True)
    top3 = df_imp.nlargest(3, "Importance")["Feature"].tolist()
    colors = [MC["blue"] if f in top3 else MC["light_blue"] for f in df_imp["Feature"]]

    fig = go.Figure(go.Bar(y=df_imp["Feature"], x=df_imp["Importance"], orientation="h",
                           marker_color=colors, text=[f"{v:.1%}" for v in df_imp["Importance"]],
                           textposition="outside", textfont=dict(size=11, color=MC["dark"])))
    mck(fig, title=dict(text="CLV feature importance — Random Forest classifier"),
        xaxis=dict(title="Gini importance", tickformat=".0%"),
        margin=dict(t=65, b=55, l=170, r=20))
    return fig


@capture("graph")
def graph_5a(data_frame):
    ct = pd.crosstab(data_frame["CLV_Tier_Bin"], data_frame["CLV_Tier_Reg"])
    ct = ct.reindex(index=["High", "Not-High"], columns=["High", "Mid", "Low"], fill_value=0)

    fig = go.Figure(go.Heatmap(
        z=ct.values, x=["High", "Mid", "Low"], y=[f"Binary: {r}" for r in ["High", "Not-High"]],
        colorscale=[[0, MC["bg"]], [0.3, MC["light_blue"]], [1.0, MC["blue"]]],
        text=[[f"<b>{v}</b>" for v in row] for row in ct.values], texttemplate="%{text}",
        textfont=dict(size=18, color=MC["dark"]),
        hovertemplate="Binary: %{y}<br>Regression: %{x}<br>Count: %{z}<extra></extra>",
        showscale=False))

    mck(fig, title=dict(text="CLV model agreement — where do the models agree?"),
        xaxis_title="Regression CLV tier", yaxis_title="",
        margin=dict(t=65, b=55, l=120, r=20))
    return fig


@capture("graph")
def graph_5b(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["CLV_High_Prob"]
        fig.add_trace(go.Histogram(x=subset, nbinsx=50, histnorm="probability density",
                                   name=f"{tier} (n={len(subset)})", marker_color=color, opacity=0.55))
    fig.add_vline(x=0.5, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text="Binary threshold", annotation_position="top right",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, barmode="overlay", title=dict(text="Binary CLV probability by regression tier"),
        xaxis=dict(title="CLV high probability", range=[-0.02, 1.05]),
        yaxis_title="Density")
    return fig


# ---- PAGE 6: ROI ----

@capture("graph")
def graph_6a(data_frame):
    ct = data_frame.groupby(["ChurnBand", "CLV_Tier_Bin"]).agg(
        Count=("CustomerID", "count"), MeanCLVProb=("CLV_High_Prob", "mean"),
        MeanChurnProb=("Churn_Prob", "mean")).reset_index()
    ct["RiskIndex"] = ct["Count"] * ct["MeanCLVProb"] * ct["MeanChurnProb"]

    fig = go.Figure()
    for tier, color in [("High", MC["blue"]), ("Not-High", MC["light_blue"])]:
        subset = ct[ct["CLV_Tier_Bin"] == tier]
        fig.add_trace(go.Bar(x=subset["ChurnBand"].astype(str), y=subset["RiskIndex"],
                             name=f"{tier} CLV", marker_color=color,
                             text=[f"{v:.0f}" for v in subset["RiskIndex"]],
                             textposition="outside", textfont=dict(size=11, color=MC["dark"])))

    key_band = ct[(ct["ChurnBand"] == "4–25%") & (ct["CLV_Tier_Bin"] == "High")]
    if len(key_band) > 0:
        fig.add_annotation(x="4–25%", y=key_band.iloc[0]["RiskIndex"] + 3,
                           text=f"<b>Core opportunity:</b><br>{int(key_band.iloc[0]['Count'])} High-CLV customers",
                           showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["blue"],
                           font=dict(size=10, color=MC["blue"]), ax=65, ay=-30)

    mck(fig, barmode="group", title=dict(text="Where is the money most at risk?"),
        xaxis_title="Churn probability band", yaxis_title="Revenue-at-risk index")
    return fig


# =============================================================================
# PAGES — Business Insights Integrated
# =============================================================================

page_1 = vm.Page(
    title="Revenue Concentration",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 1.1 — The Business Is Dangerously Concentrated",
            components=[
                vm.Graph(figure=graph_1a(data_frame="df_merged")),
                vm.Card(text="""
The top 1% of customers (~43 accounts) generate **~31.8% of total revenue**. The top 5% generate ~50.4%. \
The Gini coefficient of **0.716** confirms this is not a healthy long-tail — it is a business where a \
handful of accounts constitute an existential dependency.

**345 of 347** customers with >75% churn probability are Not-High CLV. Revenue concentration becomes \
risk concentration — you are not losing your valuable customers to churn, you are losing your low-value ones.

**Decision enabled:** The CFO must treat the top 43 accounts as a separate risk category with dedicated \
retention resource. Losing three in a single quarter is a material revenue event.
"""),
            ],
        ),
        vm.Container(
            title="Insight 1.2 — The Revenue-at-Risk Exposure Is Quantified",
            components=[
                vm.Card(text="""
The **Protect quadrant** contains 140 customers (4.2% of the base) holding **£546,537** in predicted \
lifetime value — 33.6% of total predicted CLV. At mean churn probability of 11.6%, expected unmitigated \
loss is **~£63,537**.

The break-even retention rate at £10/customer is **0.3%**. At £25 it is 0.6%. At £50 it is 1.3%. The \
programme covers its own cost if it retains fewer than **2 customers in every 150 targeted**.

Losing the top 10 customers costs **£1,537,659** (17.3% of revenue). Losing the top 50 costs £2,959,342 (33.3%).

**Decision enabled:** A programme costing £50,000 annually that prevents the loss of two top-10 accounts \
pays for itself **14× over**. The cost of the programme is not the risk. The cost of inaction is.
"""),
            ],
        ),
    ],
)

page_2 = vm.Page(
    title="Segmentation",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 2.1 — Four Segments Require Four Different Strategies",
            components=[
                vm.Graph(figure=graph_2a(data_frame="df_merged")),
                vm.Card(text="""
Every customer positioned by churn risk (x-axis) and CLV probability (y-axis). Four quadrants at \
churn threshold 0.04 (profit-optimised) and CLV probability 0.50:

**Protect** (104) — High CLV, low churn. £546K predicted CLV. Dedicated account ownership. \
**Nurture** (657) — High CLV, at risk. Primary retention target at £582K. \
**Monitor** (1) — Low CLV, safe. Digital-only. \
**Deprioritise** (2,608) — Low CLV, high churn. Zero high-touch spend justified.

**Decision enabled:** Budget allocation is proportional to CLV at stake per quadrant, not customer count.
"""),
            ],
        ),
        vm.Container(
            title="Insight 2.2 — Model Disagreement Defines the Monitoring Zone",
            components=[
                vm.Graph(figure=graph_2b(data_frame="df_merged")),
                vm.Card(text="""
Binary model: **761 High**, 2,609 Not-High. Regression model: **603 High**, 570 Mid, 2,197 Low. \
Model disagreement on **158 customers** — the honest acknowledgment of uncertainty.

**513 customers** classified High by both models are confident Protect/Nurture candidates. \
The 158 in disagreement enter a watch-list workflow with lighter-touch intervention.
"""),
            ],
        ),
    ],
)

page_3 = vm.Page(
    title="Churn Prediction",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 3.3 — The Optimal Threshold and Its Business Value",
            components=[
                vm.Graph(figure=graph_3a(data_frame="df_merged")),
                vm.Card(text="""
The default 50% threshold is the **wrong operational threshold**. At the profit-maximising threshold \
of **4%**: 3,265 flagged. Expected revenue recovered **£187,877**. Net value **£168,257**. \
ROI approximately **8.6:1** at midpoint assumptions.

**Decision enabled:** Set the CRM churn alert trigger at **0.04 probability**. The difference between \
4% and 50% is ~£168,000 in expected net value per cycle.
"""),
            ],
        ),
        vm.Container(
            title="Insight 3.1 — Almost Everyone Is At Risk (With an Honest Caveat)",
            components=[
                vm.Graph(figure=graph_3b(data_frame="df_merged")),
                vm.Card(text="""
Only **3.1%** of customers have churn probability below 0.04 — 96.9% are technically at risk. \
In non-contractual retail, this is behaviourally accurate.

**The honest caveat:** AUC-ROC of **0.755** — below the QG3 target of 0.80. The model is a ranked \
prioritisation tool, not a certainty engine. Every decision should be treated as a strong signal \
requiring human judgment.
"""),
            ],
        ),
    ],
)

page_4 = vm.Page(
    title="Why Customers Churn",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 4.1 — CLV Tier Is the Strongest Natural Separator of Churn Risk",
            components=[
                vm.Graph(figure=graph_4a(data_frame="df_merged")),
                vm.Card(text="""
High CLV median churn: **0.196**. Mid: **0.403**. Low: **0.608**. The boxes barely overlap.

Global SHAP: **Monetary** (32.7%), **Frequency** (25.0%), **AverageRevenuePerOrder** (18.5%), \
**UniqueItemsPerOrder** (11.1%), **Recency** (9.6%) — 96.8% of total signal.

**Decision enabled:** The CRM health metric is a composite led by Monetary trajectory and Frequency \
decline, not a single recency flag.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.2 — Churn Drivers Differ Materially Across Segments",
            components=[
                vm.Card(text="""
**Platinum churn** — Relationship management problem. B2B wholesale accounts with InterPurchaseInterval \
of 46 days. Signal: cycle deviation + Frequency decline. **Owner: Sales function.**

**Silver churn** — Engagement deepening problem. Recency 87 days, near-zero volatility. Trigger: \
customer crossing **60 days** post-last-purchase. **Owner: CRM & Loyalty.**

**Bronze churn** — Conversion failure. Recency 159 days, one-time buyers. Intervention window closed. \
**Owner: Low-cost digital or deprioritise entirely.**

**Decision enabled:** Route alerts by segment. Platinum/Gold → account management. Silver → CRM. \
Bronze → automated minimum-cost or no intervention.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.4 — There Is a Specific Deadline for Intervention",
            components=[
                vm.Card(text="""
**Intervention deadlines:** Platinum CRM trigger at **75 days** (10–15 day window before 90-day threshold). \
Silver trigger at **60 days**. Beyond **120 days** any segment: reclassify as win-back, lower-cost workflow.

A Platinum customer at 85 days is in the retention window. At 95 days, statistically in re-activation — \
higher cost, lower success probability.
"""),
            ],
        ),
    ],
)

page_5 = vm.Page(
    title="CLV Intelligence",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 5.1 — Regression Failed: Binary Classification Is the Fallback",
            components=[
                vm.Graph(figure=graph_5a_reg(data_frame="df_reg_models")),
                vm.Card(text="""
No model met **R² ≥ 0.50** (QG3). Ridge Train R² = 0.006. XGBoost degraded to 0.133 on holdout. \
Binary classification adopted — a **data-justified decision**.

**Decision enabled:** Remove historical Monetary as primary ranking metric. Use predicted CLV from \
the binary model for all budget allocation.
"""),
            ],
        ),
        vm.Container(
            title="Where Regression Still Matters — Top-5% Accuracy",
            components=[
                vm.Graph(figure=graph_5b_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Random Forest: **£13,600 MAE** on top-5%. Overall R² is the wrong metric when your business case \
concentrates on 5% of customers. Use the binary classifier for tier assignment; use the RF regressor \
for **within-tier ranking** of Protect accounts.
"""),
            ],
        ),
        vm.Container(
            title="Binary CLV Classifier — Production Performance",
            components=[
                vm.Graph(figure=graph_5c_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Accuracy **83.8%**. Recall **62.4%**. Precision **58.5%**. CLV threshold: training set median **£1,810**.

Per-customer spend ceilings: Protect **£780**, Nurture loyalty investment justified, Monitor **£10–£25**, \
Deprioritise **no spend above £10**.
"""),
            ],
        ),
        vm.Container(
            title="CLV Feature Importance",
            components=[
                vm.Graph(figure=graph_5d_reg(data_frame="df_reg_models")),
                vm.Card(text="""
**Monetary (31.1%)** dominates. Same 5 features drive both churn and CLV models — confirming \
feature engineering was sound. Any Protect/Nurture customer with return rate >25% needs CLV adjusted: \
CLV × (1 − return rate).
"""),
            ],
        ),
        vm.Container(
            title="Insight 5.2 — CLV Distribution Across the Priority Matrix",
            components=[
                vm.Graph(figure=graph_5a(data_frame="df_merged")),
                vm.Card(text="""
**513** classified High by both models. Protect:Deprioritise CLV ratio is **25:1**.

| Quadrant | Customers | Mean CLV | Total CLV | % Forward Rev |
|---|---|---|---|---|
| **Protect** | 140 | £3,904 | £546,537 | 33.6% |
| **Nurture** | 57 | £10,220 | £582,517 | 35.8% |
| **Deprioritise** | 3,168 | £156 | £494,953 | 30.4% |
"""),
            ],
        ),
        vm.Container(
            title="Model Consistency Check",
            components=[
                vm.Graph(figure=graph_5b(data_frame="df_merged")),
                vm.Card(text="""
High-regression-tier customers cluster at high CLV_High_Prob. Clean separation validates the \
binary fallback. Overlap in **0.3–0.7** quantifies genuine uncertainty — the 158 disagreement customers.
"""),
            ],
        ),
    ],
)

page_6 = vm.Page(
    title="ROI",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 6.1 — Where Is the Money Most at Risk?",
            components=[
                vm.Graph(figure=graph_6a(data_frame="df_merged")),
                vm.Card(text="""
**336 High-CLV customers** in the 4–25% churn band = core retention opportunity.

ROI range: **7.8:1** (conservative) to **117.1:1** (optimistic). Midpoint **31.2:1**. \
Every scenario is ROI-positive. Budget: 140 × £25 = **£3,500**. Expected preserved: **£109,307**.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.2 — The Cost of Doing Nothing Is £63,573",
            components=[
                vm.Card(text="""
Expected unmitigated loss: **£63,573** (Protect CLV £546,537 × 11.6% churn probability). \
Midpoint programme costs **£3,500**. Net expected value: **£105,807**.

Total all-tier investment: **£4,975**. Combined CLV at stake: **£1,129,054**. \
Success metric: if Protect retention exceeds **1.3%**, programme has paid for itself at highest cost.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.3 — Stop Spending on the Deprioritise Quadrant",
            components=[
                vm.Card(text="""
Deprioritise: 3,168 customers, mean CLV £156, 53.9% churn. At £25/customer and 10% retention: \
**loss of £9.38 per customer**.

Current undifferentiated spend on Deprioritise: **£79,200**. Reallocate to 140 Protect customers → \
per-customer spend rises to **£591**, funding dedicated account management. Revenue preserved unchanged \
at £109,307. **This is a reallocation instruction, not a budget increase.**
"""),
            ],
        ),
    ],
)


# =============================================================================
# DASHBOARD ASSEMBLY
# =============================================================================

dashboard = vm.Dashboard(
    title="CIRPA — Customer Intelligence & Retention Analytics",
    pages=[page_1, page_2, page_3, page_4, page_5, page_6],
    theme="vizro_light",
    navigation=vm.Navigation(
        nav_selector=vm.NavBar(
            items=[
                vm.NavLink(icon="bar_chart", label="Revenue", pages=["Revenue Concentration"]),
                vm.NavLink(icon="groups", label="Segments", pages=["Segmentation"]),
                vm.NavLink(icon="warning", label="Churn", pages=["Churn Prediction"]),
                vm.NavLink(icon="psychology", label="Drivers", pages=["Why Customers Churn"]),
                vm.NavLink(icon="payments", label="CLV", pages=["CLV Intelligence"]),
                vm.NavLink(icon="trending_up", label="ROI", pages=["ROI"]),
            ]
        )
    ),
)

Vizro().build(dashboard).run(port=8053, jupyter_mode="external")

"""
CIRPA-2026-S1 — Executive BI Dashboard (Vizro)
McKinsey & Company Visual Standard

Design language:
  - White background, minimal gridlines
  - Two-colour palette: McKinsey Blue (#027BFF) + Charcoal (#051C2C)
  - Accent: Light blue (#9FD5FF), Warm alert (#D94F00)
  - Font: Bower / Georgia (serif headings), Helvetica Neue (body)
  - Bold chart titles, direct annotations, no chart chrome
"""

import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px

import vizro.models as vm
import vizro.plotly.express as vpx
from vizro import Vizro
from vizro.managers import data_manager
from vizro.models.types import capture

# =============================================================================
# LOAD DATA
# =============================================================================

DATA_DIR = "/Users/nxmai0309/Documents/My project/Marketing_analytics/outputs"

churn = pd.read_csv(f"{DATA_DIR}/customer_churn_scores.csv")
clv_bin = pd.read_csv(f"{DATA_DIR}/customer_clv_binary_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Bin"})
clv_reg = pd.read_csv(f"{DATA_DIR}/customer_clv_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Reg"})
df_reg_models = pd.read_csv(f"{DATA_DIR}/clv_model_comparison.csv")
clv_model = joblib.load(f"{DATA_DIR}/clv_binary_model.joblib")

df = churn.merge(clv_bin, on="CustomerID").merge(clv_reg, on="CustomerID")

CHURN_BINS = [0, 0.04, 0.25, 0.50, 0.75, 1.01]
CHURN_LABELS = ["<4%", "4–25%", "25–50%", "50–75%", ">75%"]
df["ChurnBand"] = pd.cut(df["Churn_Prob"], bins=CHURN_BINS, labels=CHURN_LABELS, right=False)

TN, FP, FN, TP = 482, 59, 50, 83

data_manager["df_merged"] = df
data_manager["df_reg_models"] = df_reg_models

# =============================================================================
# McKINSEY COLOUR PALETTE
# =============================================================================

MC = {
    "blue":      "#027BFF",   # McKinsey primary blue
    "dark":      "#051C2C",   # Charcoal / near-black
    "light_blue":"#9FD5FF",   # Light accent blue
    "alert":     "#D94F00",   # Warm alert / action red-orange
    "mid_grey":  "#8C8C8C",   # Axis labels, secondary text
    "light_grey":"#D9D9D9",   # Gridlines, borders
    "bg":        "#FFFFFF",   # White background
    "text":      "#051C2C",   # Primary text
    "muted":     "#6B7280",   # Muted annotation text
}

# McKinsey chart layout template
MCK_LAYOUT = dict(
    font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", color=MC["dark"], size=12),
    title=dict(font=dict(family="Georgia, Times New Roman, serif", size=18, color=MC["dark"]), x=0, xanchor="left"),
    plot_bgcolor=MC["bg"],
    paper_bgcolor=MC["bg"],
    xaxis=dict(
        showgrid=False, zeroline=False,
        linecolor=MC["light_grey"], linewidth=1, ticks="outside", ticklen=4,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    yaxis=dict(
        showgrid=True, gridcolor=MC["light_grey"], gridwidth=0.5, griddash="dot",
        zeroline=False, showline=False,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    legend=dict(
        orientation="h", y=-0.15, x=0, xanchor="left",
        font=dict(size=11, color=MC["mid_grey"]),
        bgcolor="rgba(0,0,0,0)", borderwidth=0,
    ),
    margin=dict(t=65, b=55, l=60, r=20),
    hoverlabel=dict(bgcolor=MC["dark"], font_size=11, font_family="Helvetica Neue"),
)

def mck(fig, **overrides):
    """Apply McKinsey layout to any figure."""
    layout = {**MCK_LAYOUT, **overrides}
    fig.update_layout(**layout)
    return fig


# =============================================================================
# CHART FUNCTIONS
# =============================================================================

# ---- PAGE 1: Revenue Concentration ----

@capture("graph")
def graph_1a(data_frame):
    ct = pd.crosstab(data_frame["ChurnBand"], data_frame["CLV_Tier_Bin"])
    for col in ["High", "Not-High"]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[["High", "Not-High"]]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["High"], name="High CLV",
                         marker_color=MC["blue"], text=ct["High"],
                         textposition="inside", textfont=dict(size=12, color="white")))
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["Not-High"], name="Not-High CLV",
                         marker_color=MC["light_blue"], text=ct["Not-High"],
                         textposition="inside", textfont=dict(size=12, color=MC["dark"])))

    total_gt75 = ct.loc[">75%"].sum()
    not_high_gt75 = ct.loc[">75%", "Not-High"]
    fig.add_annotation(x=">75%", y=total_gt75 + 40,
                       text=f"<b>{not_high_gt75} of {total_gt75}</b> are Not-High CLV",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["alert"],
                       font=dict(size=11, color=MC["alert"]), ax=0, ay=-30)

    mck(fig, barmode="stack",
        title=dict(text="Churn risk by CLV tier"),
        xaxis_title="Churn probability band", yaxis_title="Number of customers")
    return fig


# ---- PAGE 2: Segmentation ----

@capture("graph")
def graph_2a(data_frame):
    churn_thresh, clv_thresh = 0.04, 0.5
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=data_frame["Churn_Prob"], y=data_frame["CLV_High_Prob"], mode="markers",
        marker=dict(size=4, opacity=0.4, color=data_frame["CLV_High_Prob"],
                    colorscale=[[0, MC["light_grey"]], [0.5, MC["light_blue"]], [1, MC["blue"]]],
                    showscale=False),
        hovertemplate="Churn: %{x:.2f}<br>CLV prob: %{y:.2f}<extra></extra>", showlegend=False))

    fig.add_hline(y=clv_thresh, line_dash="dash", line_color=MC["mid_grey"], line_width=1)
    fig.add_vline(x=churn_thresh, line_dash="dash", line_color=MC["alert"], line_width=1.5)

    protect = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    nurture = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    monitor = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()
    depri = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()

    for lx, ly, txt, col in [
        (0.02, 0.95, f"PROTECT  {protect}", MC["blue"]),
        (0.55, 0.95, f"NURTURE  {nurture}", MC["alert"]),
        (0.02, 0.05, f"MONITOR  {monitor}", MC["mid_grey"]),
        (0.55, 0.05, f"DEPRIORITISE  {depri}", MC["muted"]),
    ]:
        fig.add_annotation(x=lx, y=ly, text=f"<b>{txt}</b>", showarrow=False,
                           font=dict(size=12, color=col, family="Helvetica Neue"), xref="x", yref="y")

    mck(fig, title=dict(text="Retention priority matrix — every customer positioned"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02], showgrid=False),
        yaxis=dict(title="CLV high probability", range=[-0.02, 1.02], showgrid=False))
    return fig


@capture("graph")
def graph_2b(data_frame):
    bin_counts = data_frame["CLV_Tier_Bin"].value_counts()
    reg_counts = data_frame["CLV_Tier_Reg"].value_counts()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=["High", "Not-High"],
                         y=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         name="Binary model", marker_color=MC["blue"],
                         text=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))
    fig.add_trace(go.Bar(x=["High", "Mid", "Low"],
                         y=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         name="Regression model", marker_color=MC["light_blue"],
                         text=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))

    gap = abs(bin_counts.get("High", 0) - reg_counts.get("High", 0))
    fig.add_annotation(x=0.98, y=0.95, xref="paper", yref="paper",
                       text=f"<b>{gap}</b> customers in<br>disagreement zone",
                       showarrow=False, font=dict(size=11, color=MC["alert"]),
                       align="right")

    mck(fig, barmode="group", title=dict(text="CLV tier distribution — binary vs regression"),
        xaxis_title="CLV tier", yaxis_title="Number of customers")
    return fig


# ---- PAGE 3: Churn Prediction ----

@capture("graph")
def graph_3a(data_frame):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=data_frame["Churn_Prob"], nbinsx=60,
                               marker_color=MC["blue"], opacity=0.85, name="Customers"))

    n_opt = (data_frame["Churn_Prob"] >= 0.04).sum()
    n_def = (data_frame["Churn_Prob"] >= 0.50).sum()
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>Optimal 0.04</b><br>{n_opt:,} flagged",
                  annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"Default 0.50<br>{n_def:,} flagged",
                  annotation_position="top left",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, title=dict(text="Why the threshold is 0.04, not 0.50"),
        xaxis_title="Churn probability", yaxis_title="Number of customers", showlegend=False)
    return fig


@capture("graph")
def graph_3b(data_frame):
    sorted_probs = np.sort(data_frame["Churn_Prob"].values)
    cdf = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sorted_probs, y=cdf, mode="lines",
                             line=dict(color=MC["blue"], width=2.5), name="CDF",
                             fill="tozeroy", fillcolor="rgba(2,123,255,0.06)"))

    pct_004 = (data_frame["Churn_Prob"] < 0.04).mean() * 100
    pct_050 = (data_frame["Churn_Prob"] < 0.50).mean() * 100
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>0.04</b> → {pct_004:.1f}% below",
                  annotation_position="top right", annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"0.50 → {pct_050:.1f}% below",
                  annotation_position="top left", annotation_font=dict(color=MC["mid_grey"], size=10))
    fig.add_hline(y=pct_004, line_dash="dot", line_color=MC["alert"], line_width=1, opacity=0.4)

    mck(fig, title=dict(text="Churn probability CDF — almost everyone is at risk"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02]),
        yaxis=dict(title="% of customers (cumulative)", range=[0, 105]),
        showlegend=False)
    return fig


# ---- PAGE 4: Why Customers Churn ----

@capture("graph")
def graph_4a(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["Churn_Prob"]
        fig.add_trace(go.Box(y=subset, name=f"{tier} CLV", marker_color=color,
                             line=dict(color=MC["dark"], width=1.5), boxmean=False,
                             fillcolor=color))

    medians = {t: data_frame[data_frame["CLV_Tier_Reg"] == t]["Churn_Prob"].median() for t in tier_order}
    for i, (tier, med) in enumerate(medians.items()):
        fig.add_annotation(x=i, y=med - 0.06, text=f"<b>{med:.3f}</b>", showarrow=False,
                           font=dict(size=11, color=MC["dark"]))

    mck(fig, title=dict(text="Churn risk by CLV tier — value predicts retention"),
        yaxis=dict(title="Churn probability", range=[-0.05, 1.05]),
        xaxis_title="CLV tier (regression model)", showlegend=False)
    return fig


# ---- PAGE 5: CLV Intelligence ----

@capture("graph")
def graph_5a_reg(data_frame):
    models = data_frame["Model"].tolist()
    cv_r2 = data_frame["CV R² Mean"].tolist()
    holdout_r2 = data_frame["Holdout R²"].tolist()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=models, y=cv_r2, name="CV R² (5-fold mean)", marker_color=MC["dark"],
                         text=[f"{v:.3f}" for v in cv_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_trace(go.Bar(x=models, y=holdout_r2, name="Holdout R²", marker_color=MC["blue"],
                         text=[f"{v:.3f}" for v in holdout_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_hline(y=0.50, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text="<b>QG3 threshold (R² ≥ 0.50)</b>", annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10, family="Helvetica Neue"))

    train_r2 = data_frame.iloc[0]["Train R²"]
    overfit_gap = data_frame.iloc[0]["Overfit Gap"]
    fig.add_annotation(x="Ridge (baseline)", y=holdout_r2[0] + 0.04,
                       text=f"Train R² = {train_r2:.3f}<br>Gap: {overfit_gap:+.3f}",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["mid_grey"],
                       font=dict(size=10, color=MC["muted"]), ax=75, ay=-40)

    mck(fig, barmode="group", title=dict(text="Why regression failed — no model met R² ≥ 0.50"),
        yaxis=dict(title="R² score", range=[-0.05, 0.65], zeroline=True, zerolinecolor=MC["light_grey"]),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5b_reg(data_frame):
    models = data_frame["Model"].tolist()
    top5_mae = data_frame["Top-5% MAE"].tolist()
    min_idx = int(np.argmin(top5_mae))
    colors = [MC["light_blue"]] * len(models)
    colors[min_idx] = MC["blue"]

    fig = go.Figure(go.Bar(x=models, y=top5_mae, marker_color=colors,
                           text=[f"£{v:,.0f}" for v in top5_mae], textposition="outside",
                           textfont=dict(size=12, color=MC["dark"]), width=0.5))
    fig.add_annotation(x=models[min_idx], y=top5_mae[min_idx] - 800,
                       text="<b>✓ Lowest error</b>", showarrow=False,
                       font=dict(size=10, color=MC["blue"]))

    mck(fig, title=dict(text="Prediction error on top-5% customers (MAE)"),
        yaxis=dict(title="Mean absolute error (£)", tickformat=",.0f", tickprefix="£"),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5c_reg(data_frame):
    total = TN + FP + FN + TP
    accuracy = (TP + TN) / total
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * precision * recall / (precision + recall)

    metrics = ["Accuracy", "Recall", "Precision", "F1 Score"]
    values = [accuracy, recall, precision, f1]
    colors = [MC["blue"], MC["blue"], MC["light_blue"], MC["light_blue"]]

    fig = go.Figure(go.Bar(y=metrics, x=values, orientation="h", marker_color=colors,
                           text=[f"<b>{v:.1%}</b>" for v in values], textposition="outside",
                           textfont=dict(size=13, color=MC["dark"])))
    mck(fig, title=dict(text="Binary CLV classifier — Random Forest performance"),
        xaxis=dict(title="Score", range=[0, 1.05], tickformat=".0%"),
        margin=dict(t=65, b=55, l=95, r=20))
    return fig


@capture("graph")
def graph_5d_reg(data_frame):
    feature_names = list(clv_model.feature_names_in_)
    importances = list(clv_model.feature_importances_)
    df_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=True)
    top3 = df_imp.nlargest(3, "Importance")["Feature"].tolist()
    colors = [MC["blue"] if f in top3 else MC["light_blue"] for f in df_imp["Feature"]]

    fig = go.Figure(go.Bar(y=df_imp["Feature"], x=df_imp["Importance"], orientation="h",
                           marker_color=colors, text=[f"{v:.1%}" for v in df_imp["Importance"]],
                           textposition="outside", textfont=dict(size=11, color=MC["dark"])))
    mck(fig, title=dict(text="CLV feature importance — Random Forest classifier"),
        xaxis=dict(title="Gini importance", tickformat=".0%"),
        margin=dict(t=65, b=55, l=170, r=20))
    return fig


@capture("graph")
def graph_5a(data_frame):
    ct = pd.crosstab(data_frame["CLV_Tier_Bin"], data_frame["CLV_Tier_Reg"])
    ct = ct.reindex(index=["High", "Not-High"], columns=["High", "Mid", "Low"], fill_value=0)

    fig = go.Figure(go.Heatmap(
        z=ct.values, x=["High", "Mid", "Low"], y=[f"Binary: {r}" for r in ["High", "Not-High"]],
        colorscale=[[0, MC["bg"]], [0.3, MC["light_blue"]], [1.0, MC["blue"]]],
        text=[[f"<b>{v}</b>" for v in row] for row in ct.values], texttemplate="%{text}",
        textfont=dict(size=18, color=MC["dark"]),
        hovertemplate="Binary: %{y}<br>Regression: %{x}<br>Count: %{z}<extra></extra>",
        showscale=False))

    mck(fig, title=dict(text="CLV model agreement — where do the models agree?"),
        xaxis_title="Regression CLV tier", yaxis_title="",
        margin=dict(t=65, b=55, l=120, r=20))
    return fig


@capture("graph")
def graph_5b(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["CLV_High_Prob"]
        fig.add_trace(go.Histogram(x=subset, nbinsx=50, histnorm="probability density",
                                   name=f"{tier} (n={len(subset)})", marker_color=color, opacity=0.55))
    fig.add_vline(x=0.5, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text="Binary threshold", annotation_position="top right",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, barmode="overlay", title=dict(text="Binary CLV probability by regression tier"),
        xaxis=dict(title="CLV high probability", range=[-0.02, 1.05]),
        yaxis_title="Density")
    return fig


# ---- PAGE 6: ROI ----

@capture("graph")
def graph_6a(data_frame):
    ct = data_frame.groupby(["ChurnBand", "CLV_Tier_Bin"]).agg(
        Count=("CustomerID", "count"), MeanCLVProb=("CLV_High_Prob", "mean"),
        MeanChurnProb=("Churn_Prob", "mean")).reset_index()
    ct["RiskIndex"] = ct["Count"] * ct["MeanCLVProb"] * ct["MeanChurnProb"]

    fig = go.Figure()
    for tier, color in [("High", MC["blue"]), ("Not-High", MC["light_blue"])]:
        subset = ct[ct["CLV_Tier_Bin"] == tier]
        fig.add_trace(go.Bar(x=subset["ChurnBand"].astype(str), y=subset["RiskIndex"],
                             name=f"{tier} CLV", marker_color=color,
                             text=[f"{v:.0f}" for v in subset["RiskIndex"]],
                             textposition="outside", textfont=dict(size=11, color=MC["dark"])))

    key_band = ct[(ct["ChurnBand"] == "4–25%") & (ct["CLV_Tier_Bin"] == "High")]
    if len(key_band) > 0:
        fig.add_annotation(x="4–25%", y=key_band.iloc[0]["RiskIndex"] + 3,
                           text=f"<b>Core opportunity:</b><br>{int(key_band.iloc[0]['Count'])} High-CLV customers",
                           showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["blue"],
                           font=dict(size=10, color=MC["blue"]), ax=65, ay=-30)

    mck(fig, barmode="group", title=dict(text="Where is the money most at risk?"),
        xaxis_title="Churn probability band", yaxis_title="Revenue-at-risk index")
    return fig


# =============================================================================
# PAGES — Business Insights Integrated
# =============================================================================

page_1 = vm.Page(
    title="Revenue Concentration",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 1.1 — The Business Is Dangerously Concentrated",
            components=[
                vm.Graph(figure=graph_1a(data_frame="df_merged")),
                vm.Card(text="""
The top 1% of customers (~43 accounts) generate **~31.8% of total revenue**. The top 5% generate ~50.4%. \
The Gini coefficient of **0.716** confirms this is not a healthy long-tail — it is a business where a \
handful of accounts constitute an existential dependency.

**345 of 347** customers with >75% churn probability are Not-High CLV. Revenue concentration becomes \
risk concentration — you are not losing your valuable customers to churn, you are losing your low-value ones.

**Decision enabled:** The CFO must treat the top 43 accounts as a separate risk category with dedicated \
retention resource. Losing three in a single quarter is a material revenue event.
"""),
            ],
        ),
        vm.Container(
            title="Insight 1.2 — The Revenue-at-Risk Exposure Is Quantified",
            components=[
                vm.Card(text="""
The **Protect quadrant** contains 140 customers (4.2% of the base) holding **£546,537** in predicted \
lifetime value — 33.6% of total predicted CLV. At mean churn probability of 11.6%, expected unmitigated \
loss is **~£63,537**.

The break-even retention rate at £10/customer is **0.3%**. At £25 it is 0.6%. At £50 it is 1.3%. The \
programme covers its own cost if it retains fewer than **2 customers in every 150 targeted**.

Losing the top 10 customers costs **£1,537,659** (17.3% of revenue). Losing the top 50 costs £2,959,342 (33.3%).

**Decision enabled:** A programme costing £50,000 annually that prevents the loss of two top-10 accounts \
pays for itself **14× over**. The cost of the programme is not the risk. The cost of inaction is.
"""),
            ],
        ),
    ],
)

page_2 = vm.Page(
    title="Segmentation",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 2.1 — Four Segments Require Four Different Strategies",
            components=[
                vm.Graph(figure=graph_2a(data_frame="df_merged")),
                vm.Card(text="""
Every customer positioned by churn risk (x-axis) and CLV probability (y-axis). Four quadrants at \
churn threshold 0.04 (profit-optimised) and CLV probability 0.50:

**Protect** (104) — High CLV, low churn. £546K predicted CLV. Dedicated account ownership. \
**Nurture** (657) — High CLV, at risk. Primary retention target at £582K. \
**Monitor** (1) — Low CLV, safe. Digital-only. \
**Deprioritise** (2,608) — Low CLV, high churn. Zero high-touch spend justified.

**Decision enabled:** Budget allocation is proportional to CLV at stake per quadrant, not customer count.
"""),
            ],
        ),
        vm.Container(
            title="Insight 2.2 — Model Disagreement Defines the Monitoring Zone",
            components=[
                vm.Graph(figure=graph_2b(data_frame="df_merged")),
                vm.Card(text="""
Binary model: **761 High**, 2,609 Not-High. Regression model: **603 High**, 570 Mid, 2,197 Low. \
Model disagreement on **158 customers** — the honest acknowledgment of uncertainty.

**513 customers** classified High by both models are confident Protect/Nurture candidates. \
The 158 in disagreement enter a watch-list workflow with lighter-touch intervention.
"""),
            ],
        ),
    ],
)

page_3 = vm.Page(
    title="Churn Prediction",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 3.3 — The Optimal Threshold and Its Business Value",
            components=[
                vm.Graph(figure=graph_3a(data_frame="df_merged")),
                vm.Card(text="""
The default 50% threshold is the **wrong operational threshold**. At the profit-maximising threshold \
of **4%**: 3,265 flagged. Expected revenue recovered **£187,877**. Net value **£168,257**. \
ROI approximately **8.6:1** at midpoint assumptions.

**Decision enabled:** Set the CRM churn alert trigger at **0.04 probability**. The difference between \
4% and 50% is ~£168,000 in expected net value per cycle.
"""),
            ],
        ),
        vm.Container(
            title="Insight 3.1 — Almost Everyone Is At Risk (With an Honest Caveat)",
            components=[
                vm.Graph(figure=graph_3b(data_frame="df_merged")),
                vm.Card(text="""
Only **3.1%** of customers have churn probability below 0.04 — 96.9% are technically at risk. \
In non-contractual retail, this is behaviourally accurate.

**The honest caveat:** AUC-ROC of **0.755** — below the QG3 target of 0.80. The model is a ranked \
prioritisation tool, not a certainty engine. Every decision should be treated as a strong signal \
requiring human judgment.
"""),
            ],
        ),
    ],
)

page_4 = vm.Page(
    title="Why Customers Churn",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 4.1 — CLV Tier Is the Strongest Natural Separator of Churn Risk",
            components=[
                vm.Graph(figure=graph_4a(data_frame="df_merged")),
                vm.Card(text="""
High CLV median churn: **0.196**. Mid: **0.403**. Low: **0.608**. The boxes barely overlap.

Global SHAP: **Monetary** (32.7%), **Frequency** (25.0%), **AverageRevenuePerOrder** (18.5%), \
**UniqueItemsPerOrder** (11.1%), **Recency** (9.6%) — 96.8% of total signal.

**Decision enabled:** The CRM health metric is a composite led by Monetary trajectory and Frequency \
decline, not a single recency flag.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.2 — Churn Drivers Differ Materially Across Segments",
            components=[
                vm.Card(text="""
**Platinum churn** — Relationship management problem. B2B wholesale accounts with InterPurchaseInterval \
of 46 days. Signal: cycle deviation + Frequency decline. **Owner: Sales function.**

**Silver churn** — Engagement deepening problem. Recency 87 days, near-zero volatility. Trigger: \
customer crossing **60 days** post-last-purchase. **Owner: CRM & Loyalty.**

**Bronze churn** — Conversion failure. Recency 159 days, one-time buyers. Intervention window closed. \
**Owner: Low-cost digital or deprioritise entirely.**

**Decision enabled:** Route alerts by segment. Platinum/Gold → account management. Silver → CRM. \
Bronze → automated minimum-cost or no intervention.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.4 — There Is a Specific Deadline for Intervention",
            components=[
                vm.Card(text="""
**Intervention deadlines:** Platinum CRM trigger at **75 days** (10–15 day window before 90-day threshold). \
Silver trigger at **60 days**. Beyond **120 days** any segment: reclassify as win-back, lower-cost workflow.

A Platinum customer at 85 days is in the retention window. At 95 days, statistically in re-activation — \
higher cost, lower success probability.
"""),
            ],
        ),
    ],
)

page_5 = vm.Page(
    title="CLV Intelligence",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 5.1 — Regression Failed: Binary Classification Is the Fallback",
            components=[
                vm.Graph(figure=graph_5a_reg(data_frame="df_reg_models")),
                vm.Card(text="""
No model met **R² ≥ 0.50** (QG3). Ridge Train R² = 0.006. XGBoost degraded to 0.133 on holdout. \
Binary classification adopted — a **data-justified decision**.

**Decision enabled:** Remove historical Monetary as primary ranking metric. Use predicted CLV from \
the binary model for all budget allocation.
"""),
            ],
        ),
        vm.Container(
            title="Where Regression Still Matters — Top-5% Accuracy",
            components=[
                vm.Graph(figure=graph_5b_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Random Forest: **£13,600 MAE** on top-5%. Overall R² is the wrong metric when your business case \
concentrates on 5% of customers. Use the binary classifier for tier assignment; use the RF regressor \
for **within-tier ranking** of Protect accounts.
"""),
            ],
        ),
        vm.Container(
            title="Binary CLV Classifier — Production Performance",
            components=[
                vm.Graph(figure=graph_5c_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Accuracy **83.8%**. Recall **62.4%**. Precision **58.5%**. CLV threshold: training set median **£1,810**.

Per-customer spend ceilings: Protect **£780**, Nurture loyalty investment justified, Monitor **£10–£25**, \
Deprioritise **no spend above £10**.
"""),
            ],
        ),
        vm.Container(
            title="CLV Feature Importance",
            components=[
                vm.Graph(figure=graph_5d_reg(data_frame="df_reg_models")),
                vm.Card(text="""
**Monetary (31.1%)** dominates. Same 5 features drive both churn and CLV models — confirming \
feature engineering was sound. Any Protect/Nurture customer with return rate >25% needs CLV adjusted: \
CLV × (1 − return rate).
"""),
            ],
        ),
        vm.Container(
            title="Insight 5.2 — CLV Distribution Across the Priority Matrix",
            components=[
                vm.Graph(figure=graph_5a(data_frame="df_merged")),
                vm.Card(text="""
**513** classified High by both models. Protect:Deprioritise CLV ratio is **25:1**.

| Quadrant | Customers | Mean CLV | Total CLV | % Forward Rev |
|---|---|---|---|---|
| **Protect** | 140 | £3,904 | £546,537 | 33.6% |
| **Nurture** | 57 | £10,220 | £582,517 | 35.8% |
| **Deprioritise** | 3,168 | £156 | £494,953 | 30.4% |
"""),
            ],
        ),
        vm.Container(
            title="Model Consistency Check",
            components=[
                vm.Graph(figure=graph_5b(data_frame="df_merged")),
                vm.Card(text="""
High-regression-tier customers cluster at high CLV_High_Prob. Clean separation validates the \
binary fallback. Overlap in **0.3–0.7** quantifies genuine uncertainty — the 158 disagreement customers.
"""),
            ],
        ),
    ],
)

page_6 = vm.Page(
    title="ROI",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 6.1 — Where Is the Money Most at Risk?",
            components=[
                vm.Graph(figure=graph_6a(data_frame="df_merged")),
                vm.Card(text="""
**336 High-CLV customers** in the 4–25% churn band = core retention opportunity.

ROI range: **7.8:1** (conservative) to **117.1:1** (optimistic). Midpoint **31.2:1**. \
Every scenario is ROI-positive. Budget: 140 × £25 = **£3,500**. Expected preserved: **£109,307**.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.2 — The Cost of Doing Nothing Is £63,573",
            components=[
                vm.Card(text="""
Expected unmitigated loss: **£63,573** (Protect CLV £546,537 × 11.6% churn probability). \
Midpoint programme costs **£3,500**. Net expected value: **£105,807**.

Total all-tier investment: **£4,975**. Combined CLV at stake: **£1,129,054**. \
Success metric: if Protect retention exceeds **1.3%**, programme has paid for itself at highest cost.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.3 — Stop Spending on the Deprioritise Quadrant",
            components=[
                vm.Card(text="""
Deprioritise: 3,168 customers, mean CLV £156, 53.9% churn. At £25/customer and 10% retention: \
**loss of £9.38 per customer**.

Current undifferentiated spend on Deprioritise: **£79,200**. Reallocate to 140 Protect customers → \
per-customer spend rises to **£591**, funding dedicated account management. Revenue preserved unchanged \
at £109,307. **This is a reallocation instruction, not a budget increase.**
"""),
            ],
        ),
    ],
)


# =============================================================================
# DASHBOARD ASSEMBLY
# =============================================================================

dashboard = vm.Dashboard(
    title="CIRPA — Customer Intelligence & Retention Analytics",
    pages=[page_1, page_2, page_3, page_4, page_5, page_6],
    theme="vizro_light",
    navigation=vm.Navigation(
        nav_selector=vm.NavBar(
            items=[
                vm.NavLink(icon="bar_chart", label="Revenue", pages=["Revenue Concentration"]),
                vm.NavLink(icon="groups", label="Segments", pages=["Segmentation"]),
                vm.NavLink(icon="warning", label="Churn", pages=["Churn Prediction"]),
                vm.NavLink(icon="psychology", label="Drivers", pages=["Why Customers Churn"]),
                vm.NavLink(icon="payments", label="CLV", pages=["CLV Intelligence"]),
                vm.NavLink(icon="trending_up", label="ROI", pages=["ROI"]),
            ]
        )
    ),
)

Vizro().build(dashboard).run(port=8053, jupyter_mode="external")

In [2]:
"""
CIRPA-2026-S1 — Executive BI Dashboard (Vizro)
McKinsey & Company Visual Standard

Design language:
  - White background, minimal gridlines
  - Two-colour palette: McKinsey Blue (#027BFF) + Charcoal (#051C2C)
  - Accent: Light blue (#9FD5FF), Warm alert (#D94F00)
  - Font: Bower / Georgia (serif headings), Helvetica Neue (body)
  - Bold chart titles, direct annotations, no chart chrome
"""

import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px

import vizro.models as vm
import vizro.plotly.express as vpx
from vizro import Vizro
from vizro.managers import data_manager
from vizro.models.types import capture

# =============================================================================
# LOAD DATA
# =============================================================================

DATA_DIR = "/Users/nxmai0309/Documents/My project/Marketing_analytics/outputs"

churn = pd.read_csv(f"{DATA_DIR}/customer_churn_scores.csv")
clv_bin = pd.read_csv(f"{DATA_DIR}/customer_clv_binary_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Bin"})
clv_reg = pd.read_csv(f"{DATA_DIR}/customer_clv_scores.csv").rename(columns={"CLV_Tier": "CLV_Tier_Reg"})
df_reg_models = pd.read_csv(f"{DATA_DIR}/clv_model_comparison.csv")
clv_model = joblib.load(f"{DATA_DIR}/clv_binary_model.joblib")

df = churn.merge(clv_bin, on="CustomerID").merge(clv_reg, on="CustomerID")

CHURN_BINS = [0, 0.04, 0.25, 0.50, 0.75, 1.01]
CHURN_LABELS = ["<4%", "4–25%", "25–50%", "50–75%", ">75%"]
df["ChurnBand"] = pd.cut(df["Churn_Prob"], bins=CHURN_BINS, labels=CHURN_LABELS, right=False)

TN, FP, FN, TP = 482, 59, 50, 83

data_manager["df_merged"] = df
data_manager["df_reg_models"] = df_reg_models

# =============================================================================
# McKINSEY COLOUR PALETTE
# =============================================================================

MC = {
    "blue":      "#027BFF",   # McKinsey primary blue
    "dark":      "#051C2C",   # Charcoal / near-black
    "light_blue":"#9FD5FF",   # Light accent blue
    "alert":     "#D94F00",   # Warm alert / action red-orange
    "mid_grey":  "#8C8C8C",   # Axis labels, secondary text
    "light_grey":"#D9D9D9",   # Gridlines, borders
    "bg":        "#FFFFFF",   # White background
    "text":      "#051C2C",   # Primary text
    "muted":     "#6B7280",   # Muted annotation text
}

# McKinsey chart layout template
MCK_LAYOUT = dict(
    font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", color=MC["dark"], size=12),
    title=dict(font=dict(family="Georgia, Times New Roman, serif", size=18, color=MC["dark"]), x=0, xanchor="left"),
    plot_bgcolor=MC["bg"],
    paper_bgcolor=MC["bg"],
    xaxis=dict(
        showgrid=False, zeroline=False,
        linecolor=MC["light_grey"], linewidth=1, ticks="outside", ticklen=4,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    yaxis=dict(
        showgrid=True, gridcolor=MC["light_grey"], gridwidth=0.5, griddash="dot",
        zeroline=False, showline=False,
        tickfont=dict(size=11, color=MC["mid_grey"]),
        title_font=dict(size=12, color=MC["mid_grey"]),
    ),
    legend=dict(
        orientation="h", y=-0.15, x=0, xanchor="left",
        font=dict(size=11, color=MC["mid_grey"]),
        bgcolor="rgba(0,0,0,0)", borderwidth=0,
    ),
    margin=dict(t=65, b=55, l=60, r=20),
    hoverlabel=dict(bgcolor=MC["dark"], font_size=11, font_family="Helvetica Neue"),
)

def mck(fig, **overrides):
    """Apply McKinsey layout to any figure."""
    layout = {**MCK_LAYOUT, **overrides}
    fig.update_layout(**layout)
    return fig


# =============================================================================
# CHART FUNCTIONS
# =============================================================================

# ---- PAGE 1: Revenue Concentration ----

@capture("graph")
def graph_1a(data_frame):
    ct = pd.crosstab(data_frame["ChurnBand"], data_frame["CLV_Tier_Bin"])
    for col in ["High", "Not-High"]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[["High", "Not-High"]]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["High"], name="High CLV",
                         marker_color=MC["blue"], text=ct["High"],
                         textposition="inside", textfont=dict(size=12, color="white")))
    fig.add_trace(go.Bar(x=ct.index.astype(str), y=ct["Not-High"], name="Not-High CLV",
                         marker_color=MC["light_blue"], text=ct["Not-High"],
                         textposition="inside", textfont=dict(size=12, color=MC["dark"])))

    total_gt75 = ct.loc[">75%"].sum()
    not_high_gt75 = ct.loc[">75%", "Not-High"]
    fig.add_annotation(x=">75%", y=total_gt75 + 40,
                       text=f"<b>{not_high_gt75} of {total_gt75}</b> are Not-High CLV",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["alert"],
                       font=dict(size=11, color=MC["alert"]), ax=0, ay=-30)

    mck(fig, barmode="stack",
        title=dict(text="Churn risk by CLV tier"),
        xaxis_title="Churn probability band", yaxis_title="Number of customers")
    return fig


# ---- PAGE 2: Segmentation ----

@capture("graph")
def graph_2a(data_frame):
    churn_thresh, clv_thresh = 0.04, 0.5
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=data_frame["Churn_Prob"], y=data_frame["CLV_High_Prob"], mode="markers",
        marker=dict(size=4, opacity=0.4, color=data_frame["CLV_High_Prob"],
                    colorscale=[[0, MC["light_grey"]], [0.5, MC["light_blue"]], [1, MC["blue"]]],
                    showscale=False),
        hovertemplate="Churn: %{x:.2f}<br>CLV prob: %{y:.2f}<extra></extra>", showlegend=False))

    fig.add_hline(y=clv_thresh, line_dash="dash", line_color=MC["mid_grey"], line_width=1)
    fig.add_vline(x=churn_thresh, line_dash="dash", line_color=MC["alert"], line_width=1.5)

    protect = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    nurture = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] >= clv_thresh)).sum()
    monitor = ((data_frame["Churn_Prob"] < churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()
    depri = ((data_frame["Churn_Prob"] >= churn_thresh) & (data_frame["CLV_High_Prob"] < clv_thresh)).sum()

    for lx, ly, txt, col in [
        (0.02, 0.95, f"PROTECT  {protect}", MC["blue"]),
        (0.55, 0.95, f"NURTURE  {nurture}", MC["alert"]),
        (0.02, 0.05, f"MONITOR  {monitor}", MC["mid_grey"]),
        (0.55, 0.05, f"DEPRIORITISE  {depri}", MC["muted"]),
    ]:
        fig.add_annotation(x=lx, y=ly, text=f"<b>{txt}</b>", showarrow=False,
                           font=dict(size=12, color=col, family="Helvetica Neue"), xref="x", yref="y")

    mck(fig, title=dict(text="Retention priority matrix — every customer positioned"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02], showgrid=False),
        yaxis=dict(title="CLV high probability", range=[-0.02, 1.02], showgrid=False))
    return fig


@capture("graph")
def graph_2b(data_frame):
    bin_counts = data_frame["CLV_Tier_Bin"].value_counts()
    reg_counts = data_frame["CLV_Tier_Reg"].value_counts()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=["High", "Not-High"],
                         y=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         name="Binary model", marker_color=MC["blue"],
                         text=[bin_counts.get("High", 0), bin_counts.get("Not-High", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))
    fig.add_trace(go.Bar(x=["High", "Mid", "Low"],
                         y=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         name="Regression model", marker_color=MC["light_blue"],
                         text=[reg_counts.get("High", 0), reg_counts.get("Mid", 0), reg_counts.get("Low", 0)],
                         textposition="outside", textfont=dict(size=12, color=MC["dark"])))

    gap = abs(bin_counts.get("High", 0) - reg_counts.get("High", 0))
    fig.add_annotation(x=0.98, y=0.95, xref="paper", yref="paper",
                       text=f"<b>{gap}</b> customers in<br>disagreement zone",
                       showarrow=False, font=dict(size=11, color=MC["alert"]),
                       align="right")

    mck(fig, barmode="group", title=dict(text="CLV tier distribution — binary vs regression"),
        xaxis_title="CLV tier", yaxis_title="Number of customers")
    return fig


# ---- PAGE 3: Churn Prediction ----

@capture("graph")
def graph_3a(data_frame):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=data_frame["Churn_Prob"], nbinsx=60,
                               marker_color=MC["blue"], opacity=0.85, name="Customers"))

    n_opt = (data_frame["Churn_Prob"] >= 0.04).sum()
    n_def = (data_frame["Churn_Prob"] >= 0.50).sum()
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>Optimal 0.04</b><br>{n_opt:,} flagged",
                  annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"Default 0.50<br>{n_def:,} flagged",
                  annotation_position="top left",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, title=dict(text="Why the threshold is 0.04, not 0.50"),
        xaxis_title="Churn probability", yaxis_title="Number of customers", showlegend=False)
    return fig


@capture("graph")
def graph_3b(data_frame):
    sorted_probs = np.sort(data_frame["Churn_Prob"].values)
    cdf = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sorted_probs, y=cdf, mode="lines",
                             line=dict(color=MC["blue"], width=2.5), name="CDF",
                             fill="tozeroy", fillcolor="rgba(2,123,255,0.06)"))

    pct_004 = (data_frame["Churn_Prob"] < 0.04).mean() * 100
    pct_050 = (data_frame["Churn_Prob"] < 0.50).mean() * 100
    fig.add_vline(x=0.04, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text=f"<b>0.04</b> → {pct_004:.1f}% below",
                  annotation_position="top right", annotation_font=dict(color=MC["alert"], size=10))
    fig.add_vline(x=0.50, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text=f"0.50 → {pct_050:.1f}% below",
                  annotation_position="top left", annotation_font=dict(color=MC["mid_grey"], size=10))
    fig.add_hline(y=pct_004, line_dash="dot", line_color=MC["alert"], line_width=1, opacity=0.4)

    mck(fig, title=dict(text="Churn probability CDF — almost everyone is at risk"),
        xaxis=dict(title="Churn probability", range=[-0.02, 1.02]),
        yaxis=dict(title="% of customers (cumulative)", range=[0, 105]),
        showlegend=False)
    return fig


# ---- PAGE 4: Why Customers Churn ----

@capture("graph")
def graph_4a(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["Churn_Prob"]
        fig.add_trace(go.Box(y=subset, name=f"{tier} CLV", marker_color=color,
                             line=dict(color=MC["dark"], width=1.5), boxmean=False,
                             fillcolor=color))

    medians = {t: data_frame[data_frame["CLV_Tier_Reg"] == t]["Churn_Prob"].median() for t in tier_order}
    for i, (tier, med) in enumerate(medians.items()):
        fig.add_annotation(x=i, y=med - 0.06, text=f"<b>{med:.3f}</b>", showarrow=False,
                           font=dict(size=11, color=MC["dark"]))

    mck(fig, title=dict(text="Churn risk by CLV tier — value predicts retention"),
        yaxis=dict(title="Churn probability", range=[-0.05, 1.05]),
        xaxis_title="CLV tier (regression model)", showlegend=False)
    return fig


# ---- PAGE 5: CLV Intelligence ----

@capture("graph")
def graph_5a_reg(data_frame):
    models = data_frame["Model"].tolist()
    cv_r2 = data_frame["CV R² Mean"].tolist()
    holdout_r2 = data_frame["Holdout R²"].tolist()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=models, y=cv_r2, name="CV R² (5-fold mean)", marker_color=MC["dark"],
                         text=[f"{v:.3f}" for v in cv_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_trace(go.Bar(x=models, y=holdout_r2, name="Holdout R²", marker_color=MC["blue"],
                         text=[f"{v:.3f}" for v in holdout_r2], textposition="outside",
                         textfont=dict(size=11, color=MC["dark"])))
    fig.add_hline(y=0.50, line_dash="solid", line_color=MC["alert"], line_width=2,
                  annotation_text="<b>QG3 threshold (R² ≥ 0.50)</b>", annotation_position="top right",
                  annotation_font=dict(color=MC["alert"], size=10, family="Helvetica Neue"))

    train_r2 = data_frame.iloc[0]["Train R²"]
    overfit_gap = data_frame.iloc[0]["Overfit Gap"]
    fig.add_annotation(x="Ridge (baseline)", y=holdout_r2[0] + 0.04,
                       text=f"Train R² = {train_r2:.3f}<br>Gap: {overfit_gap:+.3f}",
                       showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["mid_grey"],
                       font=dict(size=10, color=MC["muted"]), ax=75, ay=-40)

    mck(fig, barmode="group", title=dict(text="Why regression failed — no model met R² ≥ 0.50"),
        yaxis=dict(title="R² score", range=[-0.05, 0.65], zeroline=True, zerolinecolor=MC["light_grey"]),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5b_reg(data_frame):
    models = data_frame["Model"].tolist()
    top5_mae = data_frame["Top-5% MAE"].tolist()
    min_idx = int(np.argmin(top5_mae))
    colors = [MC["light_blue"]] * len(models)
    colors[min_idx] = MC["blue"]

    fig = go.Figure(go.Bar(x=models, y=top5_mae, marker_color=colors,
                           text=[f"£{v:,.0f}" for v in top5_mae], textposition="outside",
                           textfont=dict(size=12, color=MC["dark"]), width=0.5))
    fig.add_annotation(x=models[min_idx], y=top5_mae[min_idx] - 800,
                       text="<b>✓ Lowest error</b>", showarrow=False,
                       font=dict(size=10, color=MC["blue"]))

    mck(fig, title=dict(text="Prediction error on top-5% customers (MAE)"),
        yaxis=dict(title="Mean absolute error (£)", tickformat=",.0f", tickprefix="£"),
        xaxis_title="Model")
    return fig


@capture("graph")
def graph_5c_reg(data_frame):
    total = TN + FP + FN + TP
    accuracy = (TP + TN) / total
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * precision * recall / (precision + recall)

    metrics = ["Accuracy", "Recall", "Precision", "F1 Score"]
    values = [accuracy, recall, precision, f1]
    colors = [MC["blue"], MC["blue"], MC["light_blue"], MC["light_blue"]]

    fig = go.Figure(go.Bar(y=metrics, x=values, orientation="h", marker_color=colors,
                           text=[f"<b>{v:.1%}</b>" for v in values], textposition="outside",
                           textfont=dict(size=13, color=MC["dark"])))
    mck(fig, title=dict(text="Binary CLV classifier — Random Forest performance"),
        xaxis=dict(title="Score", range=[0, 1.05], tickformat=".0%"),
        margin=dict(t=65, b=55, l=95, r=20))
    return fig


@capture("graph")
def graph_5d_reg(data_frame):
    feature_names = list(clv_model.feature_names_in_)
    importances = list(clv_model.feature_importances_)
    df_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=True)
    top3 = df_imp.nlargest(3, "Importance")["Feature"].tolist()
    colors = [MC["blue"] if f in top3 else MC["light_blue"] for f in df_imp["Feature"]]

    fig = go.Figure(go.Bar(y=df_imp["Feature"], x=df_imp["Importance"], orientation="h",
                           marker_color=colors, text=[f"{v:.1%}" for v in df_imp["Importance"]],
                           textposition="outside", textfont=dict(size=11, color=MC["dark"])))
    mck(fig, title=dict(text="CLV feature importance — Random Forest classifier"),
        xaxis=dict(title="Gini importance", tickformat=".0%"),
        margin=dict(t=65, b=55, l=170, r=20))
    return fig


@capture("graph")
def graph_5a(data_frame):
    ct = pd.crosstab(data_frame["CLV_Tier_Bin"], data_frame["CLV_Tier_Reg"])
    ct = ct.reindex(index=["High", "Not-High"], columns=["High", "Mid", "Low"], fill_value=0)

    fig = go.Figure(go.Heatmap(
        z=ct.values, x=["High", "Mid", "Low"], y=[f"Binary: {r}" for r in ["High", "Not-High"]],
        colorscale=[[0, MC["bg"]], [0.3, MC["light_blue"]], [1.0, MC["blue"]]],
        text=[[f"<b>{v}</b>" for v in row] for row in ct.values], texttemplate="%{text}",
        textfont=dict(size=18, color=MC["dark"]),
        hovertemplate="Binary: %{y}<br>Regression: %{x}<br>Count: %{z}<extra></extra>",
        showscale=False))

    mck(fig, title=dict(text="CLV model agreement — where do the models agree?"),
        xaxis_title="Regression CLV tier", yaxis_title="",
        margin=dict(t=65, b=55, l=120, r=20))
    return fig


@capture("graph")
def graph_5b(data_frame):
    tier_order = ["High", "Mid", "Low"]
    colors = [MC["blue"], MC["light_blue"], MC["light_grey"]]
    fig = go.Figure()
    for tier, color in zip(tier_order, colors):
        subset = data_frame[data_frame["CLV_Tier_Reg"] == tier]["CLV_High_Prob"]
        fig.add_trace(go.Histogram(x=subset, nbinsx=50, histnorm="probability density",
                                   name=f"{tier} (n={len(subset)})", marker_color=color, opacity=0.55))
    fig.add_vline(x=0.5, line_dash="dash", line_color=MC["mid_grey"], line_width=1.5,
                  annotation_text="Binary threshold", annotation_position="top right",
                  annotation_font=dict(color=MC["mid_grey"], size=10))

    mck(fig, barmode="overlay", title=dict(text="Binary CLV probability by regression tier"),
        xaxis=dict(title="CLV high probability", range=[-0.02, 1.05]),
        yaxis_title="Density")
    return fig


# ---- PAGE 6: ROI ----

@capture("graph")
def graph_6a(data_frame):
    ct = data_frame.groupby(["ChurnBand", "CLV_Tier_Bin"]).agg(
        Count=("CustomerID", "count"), MeanCLVProb=("CLV_High_Prob", "mean"),
        MeanChurnProb=("Churn_Prob", "mean")).reset_index()
    ct["RiskIndex"] = ct["Count"] * ct["MeanCLVProb"] * ct["MeanChurnProb"]

    fig = go.Figure()
    for tier, color in [("High", MC["blue"]), ("Not-High", MC["light_blue"])]:
        subset = ct[ct["CLV_Tier_Bin"] == tier]
        fig.add_trace(go.Bar(x=subset["ChurnBand"].astype(str), y=subset["RiskIndex"],
                             name=f"{tier} CLV", marker_color=color,
                             text=[f"{v:.0f}" for v in subset["RiskIndex"]],
                             textposition="outside", textfont=dict(size=11, color=MC["dark"])))

    key_band = ct[(ct["ChurnBand"] == "4–25%") & (ct["CLV_Tier_Bin"] == "High")]
    if len(key_band) > 0:
        fig.add_annotation(x="4–25%", y=key_band.iloc[0]["RiskIndex"] + 3,
                           text=f"<b>Core opportunity:</b><br>{int(key_band.iloc[0]['Count'])} High-CLV customers",
                           showarrow=True, arrowhead=0, arrowwidth=1.5, arrowcolor=MC["blue"],
                           font=dict(size=10, color=MC["blue"]), ax=65, ay=-30)

    mck(fig, barmode="group", title=dict(text="Where is the money most at risk?"),
        xaxis_title="Churn probability band", yaxis_title="Revenue-at-risk index")
    return fig


# =============================================================================
# PAGES — Business Insights Integrated
# =============================================================================

# Layout for containers with Graph + Card: graph gets 80%, card gets 20%
GRAPH_CARD_LAYOUT = vm.Grid(grid=[[0],[0],[0],[0],[1]])

page_1 = vm.Page(
    title="Revenue Concentration",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 1.1 — The Business Is Dangerously Concentrated",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_1a(data_frame="df_merged")),
                vm.Card(text="""
The top 1% of customers (~43 accounts) generate **~31.8% of total revenue**. The top 5% generate ~50.4%. \
The Gini coefficient of **0.716** confirms this is not a healthy long-tail — it is a business where a \
handful of accounts constitute an existential dependency.

**345 of 347** customers with >75% churn probability are Not-High CLV. Revenue concentration becomes \
risk concentration — you are not losing your valuable customers to churn, you are losing your low-value ones.

**Decision enabled:** The CFO must treat the top 43 accounts as a separate risk category with dedicated \
retention resource. Losing three in a single quarter is a material revenue event.
"""),
            ],
        ),
        vm.Container(
            title="Insight 1.2 — The Revenue-at-Risk Exposure Is Quantified",
            components=[
                vm.Card(text="""
The **Protect quadrant** contains 140 customers (4.2% of the base) holding **£546,537** in predicted \
lifetime value — 33.6% of total predicted CLV. At mean churn probability of 11.6%, expected unmitigated \
loss is **~£63,537**.

The break-even retention rate at £10/customer is **0.3%**. At £25 it is 0.6%. At £50 it is 1.3%. The \
programme covers its own cost if it retains fewer than **2 customers in every 150 targeted**.

Losing the top 10 customers costs **£1,537,659** (17.3% of revenue). Losing the top 50 costs £2,959,342 (33.3%).

**Decision enabled:** A programme costing £50,000 annually that prevents the loss of two top-10 accounts \
pays for itself **14× over**. The cost of the programme is not the risk. The cost of inaction is.
"""),
            ],
        ),
    ],
)

page_2 = vm.Page(
    title="Segmentation",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 2.1 — Four Segments Require Four Different Strategies",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_2a(data_frame="df_merged")),
                vm.Card(text="""
Every customer positioned by churn risk (x-axis) and CLV probability (y-axis). Four quadrants at \
churn threshold 0.04 (profit-optimised) and CLV probability 0.50:

**Protect** (104) — High CLV, low churn. £546K predicted CLV. Dedicated account ownership. \
**Nurture** (657) — High CLV, at risk. Primary retention target at £582K. \
**Monitor** (1) — Low CLV, safe. Digital-only. \
**Deprioritise** (2,608) — Low CLV, high churn. Zero high-touch spend justified.

**Decision enabled:** Budget allocation is proportional to CLV at stake per quadrant, not customer count.
"""),
            ],
        ),
        vm.Container(
            title="Insight 2.2 — Model Disagreement Defines the Monitoring Zone",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_2b(data_frame="df_merged")),
                vm.Card(text="""
Binary model: **761 High**, 2,609 Not-High. Regression model: **603 High**, 570 Mid, 2,197 Low. \
Model disagreement on **158 customers** — the honest acknowledgment of uncertainty.

**513 customers** classified High by both models are confident Protect/Nurture candidates. \
The 158 in disagreement enter a watch-list workflow with lighter-touch intervention.
"""),
            ],
        ),
    ],
)

page_3 = vm.Page(
    title="Churn Prediction",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 3.3 — The Optimal Threshold and Its Business Value",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_3a(data_frame="df_merged")),
                vm.Card(text="""
The default 50% threshold is the **wrong operational threshold**. At the profit-maximising threshold \
of **4%**: 3,265 flagged. Expected revenue recovered **£187,877**. Net value **£168,257**. \
ROI approximately **8.6:1** at midpoint assumptions.

**Decision enabled:** Set the CRM churn alert trigger at **0.04 probability**. The difference between \
4% and 50% is ~£168,000 in expected net value per cycle.
"""),
            ],
        ),
        vm.Container(
            title="Insight 3.1 — Almost Everyone Is At Risk (With an Honest Caveat)",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_3b(data_frame="df_merged")),
                vm.Card(text="""
Only **3.1%** of customers have churn probability below 0.04 — 96.9% are technically at risk. \
In non-contractual retail, this is behaviourally accurate.

**The honest caveat:** AUC-ROC of **0.755** — below the QG3 target of 0.80. The model is a ranked \
prioritisation tool, not a certainty engine. Every decision should be treated as a strong signal \
requiring human judgment.
"""),
            ],
        ),
    ],
)

page_4 = vm.Page(
    title="Why Customers Churn",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 4.1 — CLV Tier Is the Strongest Natural Separator of Churn Risk",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_4a(data_frame="df_merged")),
                vm.Card(text="""
High CLV median churn: **0.196**. Mid: **0.403**. Low: **0.608**. The boxes barely overlap.

Global SHAP: **Monetary** (32.7%), **Frequency** (25.0%), **AverageRevenuePerOrder** (18.5%), \
**UniqueItemsPerOrder** (11.1%), **Recency** (9.6%) — 96.8% of total signal.

**Decision enabled:** The CRM health metric is a composite led by Monetary trajectory and Frequency \
decline, not a single recency flag.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.2 — Churn Drivers Differ Materially Across Segments",
            components=[
                vm.Card(text="""
**Platinum churn** — Relationship management problem. B2B wholesale accounts with InterPurchaseInterval \
of 46 days. Signal: cycle deviation + Frequency decline. **Owner: Sales function.**

**Silver churn** — Engagement deepening problem. Recency 87 days, near-zero volatility. Trigger: \
customer crossing **60 days** post-last-purchase. **Owner: CRM & Loyalty.**

**Bronze churn** — Conversion failure. Recency 159 days, one-time buyers. Intervention window closed. \
**Owner: Low-cost digital or deprioritise entirely.**

**Decision enabled:** Route alerts by segment. Platinum/Gold → account management. Silver → CRM. \
Bronze → automated minimum-cost or no intervention.
"""),
            ],
        ),
        vm.Container(
            title="Insight 4.4 — There Is a Specific Deadline for Intervention",
            components=[
                vm.Card(text="""
**Intervention deadlines:** Platinum CRM trigger at **75 days** (10–15 day window before 90-day threshold). \
Silver trigger at **60 days**. Beyond **120 days** any segment: reclassify as win-back, lower-cost workflow.

A Platinum customer at 85 days is in the retention window. At 95 days, statistically in re-activation — \
higher cost, lower success probability.
"""),
            ],
        ),
    ],
)

page_5 = vm.Page(
    title="CLV Intelligence",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 5.1 — Regression Failed: Binary Classification Is the Fallback",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5a_reg(data_frame="df_reg_models")),
                vm.Card(text="""
No model met **R² ≥ 0.50** (QG3). Ridge Train R² = 0.006. XGBoost degraded to 0.133 on holdout. \
Binary classification adopted — a **data-justified decision**.

**Decision enabled:** Remove historical Monetary as primary ranking metric. Use predicted CLV from \
the binary model for all budget allocation.
"""),
            ],
        ),
        vm.Container(
            title="Where Regression Still Matters — Top-5% Accuracy",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5b_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Random Forest: **£13,600 MAE** on top-5%. Overall R² is the wrong metric when your business case \
concentrates on 5% of customers. Use the binary classifier for tier assignment; use the RF regressor \
for **within-tier ranking** of Protect accounts.
"""),
            ],
        ),
        vm.Container(
            title="Binary CLV Classifier — Production Performance",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5c_reg(data_frame="df_reg_models")),
                vm.Card(text="""
Accuracy **83.8%**. Recall **62.4%**. Precision **58.5%**. CLV threshold: training set median **£1,810**.

Per-customer spend ceilings: Protect **£780**, Nurture loyalty investment justified, Monitor **£10–£25**, \
Deprioritise **no spend above £10**.
"""),
            ],
        ),
        vm.Container(
            title="CLV Feature Importance",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5d_reg(data_frame="df_reg_models")),
                vm.Card(text="""
**Monetary (31.1%)** dominates. Same 5 features drive both churn and CLV models — confirming \
feature engineering was sound. Any Protect/Nurture customer with return rate >25% needs CLV adjusted: \
CLV × (1 − return rate).
"""),
            ],
        ),
        vm.Container(
            title="Insight 5.2 — CLV Distribution Across the Priority Matrix",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5a(data_frame="df_merged")),
                vm.Card(text="""
**513** classified High by both models. Protect:Deprioritise CLV ratio is **25:1**.

| Quadrant | Customers | Mean CLV | Total CLV | % Forward Rev |
|---|---|---|---|---|
| **Protect** | 140 | £3,904 | £546,537 | 33.6% |
| **Nurture** | 57 | £10,220 | £582,517 | 35.8% |
| **Deprioritise** | 3,168 | £156 | £494,953 | 30.4% |
"""),
            ],
        ),
        vm.Container(
            title="Model Consistency Check",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_5b(data_frame="df_merged")),
                vm.Card(text="""
High-regression-tier customers cluster at high CLV_High_Prob. Clean separation validates the \
binary fallback. Overlap in **0.3–0.7** quantifies genuine uncertainty — the 158 disagreement customers.
"""),
            ],
        ),
    ],
)

page_6 = vm.Page(
    title="ROI",
    layout=vm.Flex(),
    components=[
        vm.Container(
            title="Insight 6.1 — Where Is the Money Most at Risk?",
            layout=GRAPH_CARD_LAYOUT,
            components=[
                vm.Graph(figure=graph_6a(data_frame="df_merged")),
                vm.Card(text="""
**336 High-CLV customers** in the 4–25% churn band = core retention opportunity.

ROI range: **7.8:1** (conservative) to **117.1:1** (optimistic). Midpoint **31.2:1**. \
Every scenario is ROI-positive. Budget: 140 × £25 = **£3,500**. Expected preserved: **£109,307**.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.2 — The Cost of Doing Nothing Is £63,573",
            components=[
                vm.Card(text="""
Expected unmitigated loss: **£63,573** (Protect CLV £546,537 × 11.6% churn probability). \
Midpoint programme costs **£3,500**. Net expected value: **£105,807**.

Total all-tier investment: **£4,975**. Combined CLV at stake: **£1,129,054**. \
Success metric: if Protect retention exceeds **1.3%**, programme has paid for itself at highest cost.
"""),
            ],
        ),
        vm.Container(
            title="Insight 6.3 — Stop Spending on the Deprioritise Quadrant",
            components=[
                vm.Card(text="""
Deprioritise: 3,168 customers, mean CLV £156, 53.9% churn. At £25/customer and 10% retention: \
**loss of £9.38 per customer**.

Current undifferentiated spend on Deprioritise: **£79,200**. Reallocate to 140 Protect customers → \
per-customer spend rises to **£591**, funding dedicated account management. Revenue preserved unchanged \
at £109,307. **This is a reallocation instruction, not a budget increase.**
"""),
            ],
        ),
    ],
)


# =============================================================================
# DASHBOARD ASSEMBLY
# =============================================================================

dashboard = vm.Dashboard(
    title="CIRPA — Customer Intelligence & Retention Analytics",
    pages=[page_1, page_2, page_3, page_4, page_5, page_6],
    theme="vizro_light",
    navigation=vm.Navigation(
        nav_selector=vm.NavBar(
            items=[
                vm.NavLink(icon="bar_chart", label="Revenue", pages=["Revenue Concentration"]),
                vm.NavLink(icon="groups", label="Segments", pages=["Segmentation"]),
                vm.NavLink(icon="warning", label="Churn", pages=["Churn Prediction"]),
                vm.NavLink(icon="psychology", label="Drivers", pages=["Why Customers Churn"]),
                vm.NavLink(icon="payments", label="CLV", pages=["CLV Intelligence"]),
                vm.NavLink(icon="trending_up", label="ROI", pages=["ROI"]),
            ]
        )
    ),
)
Vizro().build(dashboard).run(port=8053, jupyter_mode="external")

Dash app running on http://127.0.0.1:8053/
